# Session6 + XGBoost rich. Пять утренних стартов против всех получасов

Цель — снизить WAPE формы при фиксированном объёме и проверить, приблизит ли затем калибровка итог к 0.318. В обоих вариантах заменяется 25% вклада LightGBM в Session6, то есть XGBoost занимает примерно 12.5% всей смеси. Признаки, параметры, число деревьев и KMeans с 8 кластерами одинаковы. В новой ветке обучающие старты ограничены 09:30, 10:00, 10:30, 11:00 и 11:30. Оценка обеих веток идёт из 10:30 на восемь горизонтов.

Подключить в Kaggle `cleaning_comparison_inputs.zip` из `best/` и `cluster_experiment_4741653856669b1e.zip` из прогона XGBoost с метамоделью. ZIP-файлы должны быть доступны как входные файлы. Из первого берутся train/test parquet, параметры LightGBM и архив Session6. Из второго читаются только сохранённые raw OOF-прогнозы XGBoost со всех стартов. Включить GPU и выполнить Run All. XGBoost обучается на CUDA QuantileDMatrix, CatBoost базы и поправки тоже на GPU. LightGBM восстанавливается на CPU по рецепту Session6.

Для val восстанавливаем сопоставимый Session6 по его рецепту. CatBoost и LightGBM для этой ветки обучаются на целях до 27 сентября. Прогнозы 27 сентября–3 октября нужны для первого fit остатка. Для недель 4–10 и 11–17 октября остаточные модели обучаются только до начала своей недели. EWMA объёма учитывает лишь уже доступные факты с запасом четыре часа. Так архивные оценки калибратора на его собственном train не используются для выбора.

Победитель формы определяется по средней WAPE смеси на двух неделях val при одинаковых объёмах. После этого подбирается сила усадки единого множителя объёма к 1 по WAPE + |bias|. Множитель оценивается на первой неделе, кандидаты проверяются на второй. Выбранная сила применяется к коэффициенту, оценённому по обеим неделям, и фиксируется до test. На локальном test 18–31 октября обе формы оцениваются с одним архивным Session6, который использует исходный причинный протокол объёма. Восстановленный Session6 на val и архивный Session6 на test различаются обучающей историей, поэтому перенос калибровки проверяется отдельно. Параметры фиксируются до чтения прогнозов test, период ранее использовался для диагностики.

Таблицы содержат WAPE, signed bias и конкурсный score по этапам, дням и горизонтам. Итоговый ZIP включает модели новой ветки, восстановленный контроль, построчные прогнозы, протокол и таблицы. Этот эксперимент оценивает локальный test до полного переобучения. Для конкурсного сабмита нужен отдельный полный refit выбранных компонентов.


In [ ]:
TRAIN_PATH = '/kaggle/input/datasets/iskanderyamalov/case-rwb/train_solo_track.parquet'
TEST_PATH = '/kaggle/input/datasets/iskanderyamalov/case-rwb/test_solo_track.parquet'
OUTPUT_ROOT = '/kaggle/working'
CLUSTERS = (1, 8)
HORIZONS = tuple(range(1, 9))
MIN_CLUSTER_ROUTES = 10
CAT_CONFIGS = [dict(depth=6, iterations=600, learning_rate=0.05)]
RIDGE_LAMBDAS = (0.01,)
VAL_DAYS = 28
TEST_DAYS = 14
CAL_FIT_DAYS = 21
SCALE_BOUNDS = (0.8, 1.25)
TASK_TYPE = 'GPU'
THREADS = 4
PREDICT_BATCH = 100000
SEED = 42
CREATE_ARCHIVE = True
PIPELINE_VERSION = f'cat_calibration_v2_groups_fit{CAL_FIT_DAYS}_bounds{SCALE_BOUNDS}'
TARGET = 'target_1h'
STATUSES = [f'status_{i}' for i in range(1, 7)]


PARAMETERS = {'selected_xgb': {'1': {'params': {'max_depth': 6, 'learning_rate': 0.036760715868196514, 'min_child_weight': 1.0809355847970479, 'subsample': 0.9003563537429966, 'colsample_bytree': 0.6639976846136139, 'reg_alpha': 9.305271303855188e-05, 'reg_lambda': 7.596418498005285}, 'trees': 2530}, '2': {'params': {'max_depth': 4, 'learning_rate': 0.10190438495585248, 'min_child_weight': 1.8389812995413264, 'subsample': 0.6817515181500764, 'colsample_bytree': 0.6619351860505892, 'reg_alpha': 0.0022965441587741537, 'reg_lambda': 14.08596091970978}, 'trees': 1419}, '3': {'params': {'max_depth': 3, 'learning_rate': 0.07005359250881395, 'min_child_weight': 2.2064897603579636, 'subsample': 0.8779096005107611, 'colsample_bytree': 0.7846752865583575, 'reg_alpha': 0.018954349619526665, 'reg_lambda': 1.043977626067855}, 'trees': 1930}, '4': {'params': {'max_depth': 4, 'learning_rate': 0.02925325002470326, 'min_child_weight': 1.5841239345284504, 'subsample': 0.8126045473410459, 'colsample_bytree': 0.7066552862291448, 'reg_alpha': 2.3192129195348187e-05, 'reg_lambda': 0.22202924731915144}, 'trees': 1827}, '5': {'params': {'max_depth': 3, 'learning_rate': 0.03164978915749773, 'min_child_weight': 1.9902815543275585, 'subsample': 0.6844098448217815, 'colsample_bytree': 0.7432566543439275, 'reg_alpha': 0.010252186188681373, 'reg_lambda': 0.5427730704771981}, 'trees': 1387}, '6': {'params': {'max_depth': 3, 'learning_rate': 0.023401679328514332, 'min_child_weight': 3.1806793320017555, 'subsample': 0.87035767368126, 'colsample_bytree': 0.7980364720685608, 'reg_alpha': 0.02123901077609503, 'reg_lambda': 0.3328783942599951}, 'trees': 1566}, '7': {'params': {'max_depth': 5, 'learning_rate': 0.0330759240922086, 'min_child_weight': 3.8692684642092345, 'subsample': 0.6794525496919193, 'colsample_bytree': 0.6799389555259473, 'reg_alpha': 2.7314479492664567, 'reg_lambda': 0.3796668381307971}, 'trees': 1136}, '8': {'params': {'max_depth': 4, 'learning_rate': 0.09417045025581995, 'min_child_weight': 13.418857993305474, 'subsample': 0.689895840619612, 'colsample_bytree': 0.6918995986906583, 'reg_alpha': 0.3746751674810787, 'reg_lambda': 1.8540896283895405}, 'trees': 534}}}
EXPERIMENTS = [{'name': 'xgboost_rich_calibration', 'algorithm': 'xgb', 'residual': False, 'parameter_set': 'selected_xgb', 'loss': 'reg:absoluteerror', 'compact': False}]


RICH_TRAINING_SOURCE = "def fit_model(train, segmentation, family, k, settings):\n    check_deadline()\n    scales = segmentation['scale'][train['route_index']]\n    target = train['y'] / scales\n    if ACTIVE['residual']:\n        target = target - seasonal_base(train['x'], train['names'])\n    weights = scales / scales.mean()\n    if ACTIVE.get('time_half_life'):\n        age=(TIMES[train['origins']].max()-TIMES[train['origins']]).total_seconds().to_numpy()/86400\n        weights *= np.repeat(np.exp2(-age/ACTIVE['time_half_life']),len(ROUTES))\n        weights /= weights.mean()\n    labels = segmentation['labels'][k][train['route_index']]\n    params = dict(PARAMETERS[ACTIVE['parameter_set']][str(ACTIVE_H)]['params'])\n    trees = PARAMETERS[ACTIVE['parameter_set']][str(ACTIVE_H)]['trees']\n    if ACTIVE.get('compact'):\n        trees = min(trees, 900)\n        key = 'depth' if ACTIVE['algorithm'] == 'cat' else 'max_depth'\n        params[key] = min(params[key], 6)\n    if ACTIVE.get('fixed_params'):\n        params=dict(ACTIVE['fixed_params'])\n        trees=int(ACTIVE['fixed_trees'])\n    models = {}\n    for group in np.unique(labels):\n        check_deadline()\n        select = np.ones(len(labels), dtype=bool) if group == -1 else labels == group\n        started = time.monotonic()\n        if ACTIVE['algorithm'] == 'cat':\n            model = CatBoostRegressor(**params, iterations=trees, loss_function=ACTIVE['loss'],\n                                      task_type='GPU', devices='0', thread_count=THREADS,\n                                      random_seed=SEED, verbose=False, allow_writing_files=False,\n                                      allow_const_label=True, bootstrap_type='Bayesian', cat_features=['route_code'])\n        else:\n            model = xgb.XGBRegressor(**params, n_estimators=trees, objective=ACTIVE['loss'],\n                                     tree_method='hist', device='cuda:0', n_jobs=THREADS, random_state=SEED, enable_categorical=True, max_cat_to_onehot=1, eval_metric='mae')\n        model.fit(model_frame(train['x'][select],train['names'],ACTIVE['algorithm']), target[select], sample_weight=weights[select])\n        if ACTIVE['algorithm'] == 'xgb':\n            device = json.loads(model.get_booster().save_config())['learner']['generic_param']['device']\n            if not device.startswith('cuda'):\n                raise RuntimeError('XGBoost переключился на CPU')\n        models[int(group)] = dict(model=model, scaler=None)\n        print(ACTIVE['name'], 'h', ACTIVE_H, 'k', k, 'group', int(group), 'seconds', round(time.monotonic()-started, 1), flush=True)\n    return dict(models=models, k=k, residual=ACTIVE['residual'], feature_names=train['names'], experiment=ACTIVE)"

RICH_SOURCE = "import pyarrow as pa\nimport pyarrow.parquet as pq\ntarget = values[TARGET]\n\ndef past(array, positions):\n    positions = np.asarray(positions, dtype=np.int64)\n    valid = (positions >= 0) & (positions < len(array))\n    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)\n    result[~valid] = np.nan\n    return result\n\ndef average_available(matrices):\n    total = np.zeros_like(matrices[0], dtype=np.float32)\n    count = np.zeros_like(total)\n    for matrix in matrices:\n        valid = np.isfinite(matrix)\n        total += np.where(valid, matrix, 0)\n        count += valid\n    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)\n\ndef rolling(array, steps):\n    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)\nmeans = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}\nlevel24 = rolling(target, 48)\npositions = np.arange(len(times))\nextra_signals = {}\nfor name in STATUSES[:3]:\n    extra_signals[name + '_hourly'] = values[name] + past(values[name], positions - 1)\nextra_signals['warehouse_total_load'] = sum((extra_signals[name + '_hourly'] for name in STATUSES[:3]))\nextra_signals['warehouse_processing_rate'] = extra_signals['status_3_hourly'] / (extra_signals['status_1_hourly'] + np.float32(1e-05))\nfor name in STATUSES[3:]:\n    extra_signals[name + '_diff_hourly'] = values[name] - past(values[name], positions - 2)\nextra_signals['hub_buffer'] = values['status_5'] - values['status_4']\nextra_signals['hub_gate_bottleneck'] = extra_signals['status_4_diff_hourly'] - extra_signals['status_6_diff_hourly']\nextra_signals['target_acceleration'] = past(target, positions - 1) - past(target, positions - 2)\nhub_signal = extra_signals['status_6_diff_hourly']\n\ndef common_features(origins):\n    common = {}\n    for k in (0, 1, 2, 4):\n        common[f'target_back_{k}'] = past(target, origins - k)\n    for name, array in values.items():\n        for steps in (4, 12):\n            common[f'{name}_mean_{steps}'] = means[name, steps][origins]\n        if name in STATUSES:\n            common[f'{name}_latest'] = array[origins]\n            if name in STATUSES[:3]:\n                common[f'{name}_change_1h'] = array[origins] - past(array, origins - 2)\n    for name, array in extra_signals.items():\n        common[name] = array[origins]\n    for lag in (48, 96):\n        common[f'target_1h_lag_{lag}'] = past(target, origins - lag)\n    common['warehouse_total_load_lag_48'] = past(extra_signals['warehouse_total_load'], origins - 48)\n    common['status_6_diff_hourly_lag_48'] = past(hub_signal, origins - 48)\n    common['target_vs_mean_6h'] = target[origins] - means[TARGET, 12][origins]\n    recent = means[TARGET, 4][origins]\n    usual = average_available([past(means[TARGET, 4], origins - day * 48) for day in range(1, 8)])\n    common['morning_level_ratio'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)\n    return common\nCORRELATION_COLUMNS = ['selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available']\nFEATURE_GROUPS = {'calendar': ['hour', 'minute', 'hour_float', 'hour_sin', 'hour_cos', 'dayofweek', 'is_weekend', 'weekday_sin', 'weekday_cos'], 'local_statuses': ['status_1_hourly', 'status_2_hourly', 'status_3_hourly', 'warehouse_total_load', 'warehouse_processing_rate'], 'hub_combinations': ['status_4_diff_hourly', 'status_5_diff_hourly', 'status_6_diff_hourly', 'hub_buffer', 'hub_gate_bottleneck'], 'historical_lags': ['target_1h_lag_48', 'target_1h_lag_96', 'warehouse_total_load_lag_48', 'status_6_diff_hourly_lag_48'], 'target_change': ['target_acceleration'], 'correlation_lag': CORRELATION_COLUMNS, 'pilot_start_time': ['start_sin', 'start_cos'], 'pilot_peaks': ['peak7_mean_max', 'peak7_mean_level', 'peak7_share_-1', 'peak7_share_0', 'peak7_share_1']}\ndef calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):\n    grid = pd.DatetimeIndex(timestamps)\n    route_ids = np.asarray(route_ids)\n    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))\n    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):\n        raise ValueError('Ожидаются два массива [время, маршрут] одной формы')\n    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):\n        raise ValueError('Для выбора лага нужна упорядоченная получасовая сетка без дубликатов')\n    if len(np.unique(route_ids)) != len(route_ids):\n        raise ValueError('Идентификаторы маршрутов повторяются')\n    steps = float(max_lookback_hours) * 2\n    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):\n        raise ValueError('Граница поиска должна задавать целое число получасов, не меньше двух')\n    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:\n        raise ValueError('min_pairs должен быть целым числом >=3')\n    if not 0 <= min_correlation <= 1 or route_block_size < 1:\n        raise ValueError('Некорректные параметры выбора корреляции')\n    max_lag = int(steps)\n    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side='left'))\n    anchors = np.arange(max_lag + 2, stop)\n    best_corr = np.full(len(route_ids), np.nan)\n    best_lag = np.full(len(route_ids), np.nan)\n    best_pairs = np.zeros(len(route_ids), dtype=np.int32)\n    for start in range(0, len(route_ids), route_block_size):\n        finish = min(start + route_block_size, len(route_ids))\n        x = target_values[anchors, start:finish].astype(np.float64)\n        for lag in range(2, max_lag + 1):\n            z = status6_diff[anchors - lag, start:finish].astype(np.float64)\n            valid = np.isfinite(x) & np.isfinite(z)\n            count = valid.sum(axis=0)\n            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))\n            divisor = np.maximum(count, 1)\n            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))\n            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor\n            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)\n            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)\n            denominator = np.sqrt(vx * vz)\n            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))\n            corr = np.clip(corr, -1, 1)\n            previous = best_corr[start:finish]\n            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))\n            best_corr[start:finish][better] = corr[better]\n            best_lag[start:finish][better] = lag\n            best_pairs[start:finish][better] = count[better]\n    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)\n    return pd.DataFrame({'route_id': route_ids, 'selected_status6_lag_steps': np.where(accepted, best_lag, np.nan), 'selected_status6_correlation': best_corr, 'selected_status6_pair_count': best_pairs, 'status6_lag_valid': accepted.astype(np.int8)})\n\ndef add_correlation_features(frame, h, lag_table, chunk_rows=200000):\n    if h not in range(1, 9) or chunk_rows < 1:\n        raise ValueError('Некорректный горизонт или размер блока')\n    by_route = lag_table.set_index('route_id').reindex(routes)\n    route_index = pd.Index(routes)\n    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)\n    quality = {name: by_route[name].to_numpy(dtype=np.float32) for name in CORRELATION_COLUMNS[:4]}\n    for name in CORRELATION_COLUMNS:\n        frame[name] = np.full(len(frame), np.nan, dtype=np.float32)\n    column_positions = {name: frame.columns.get_loc(name) for name in CORRELATION_COLUMNS}\n    for start in range(0, len(frame), chunk_rows):\n        finish = min(start + chunk_rows, len(frame))\n        block = frame.iloc[start:finish]\n        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))\n        route = route_index.get_indexer(block.route_id)\n        if (origin < 0).any() or (route < 0).any():\n            raise ValueError('Строка признаков не сопоставлена с историей')\n        selected = lag[route]\n        source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64)\n        available = np.isfinite(selected) & (selected >= h) & (source >= 0) & (source <= origin) & (source < len(times))\n        signal = hub_signal[np.clip(source, 0, len(times) - 1), route].astype(np.float32)\n        available &= np.isfinite(signal)\n        for name, array in quality.items():\n            frame.iloc[start:finish, column_positions[name]] = array[route]\n        frame.iloc[start:finish, column_positions['lagged_status6_signal']] = np.where(available, signal, np.nan)\n        frame.iloc[start:finish, column_positions['lagged_status6_signal_available']] = available.astype(np.float32)\n    return frame\ndef horizon_dataset(h, origins):\n    slot = origins + h\n    dates = times[origins] + h * STEP\n    frame = pd.DataFrame({'cutoff': times[origins].repeat(len(routes)), 'timestamp': dates.repeat(len(routes)), 'route_id': np.tile(routes, len(origins))})\n    for name, matrix in common_features(origins).items():\n        frame[name] = matrix.ravel()\n    calendar = {'hour': dates.hour, 'minute': dates.minute, 'hour_float': dates.hour + dates.minute / 60.0, 'dayofweek': dates.dayofweek, 'is_weekend': (dates.dayofweek >= 5).astype(np.int8)}\n    calendar['hour_sin'] = np.sin(2 * np.pi * calendar['hour_float'] / 24)\n    calendar['hour_cos'] = np.cos(2 * np.pi * calendar['hour_float'] / 24)\n    calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)\n    calendar['weekday_cos'] = np.cos(2 * np.pi * dates.dayofweek / 7)\n    for name, array in calendar.items():\n        frame[name] = np.repeat(array, len(routes))\n    frame = add_correlation_features(frame, h, validation_lag_table)\n    for days in (1, 2, 7):\n        frame[f'slot_days_back_{days}'] = past(target, slot - days * 48).ravel()\n    for name, days in [('slot_daily7', range(1, 8)), ('slot_weekly4', (7, 14, 21, 28))]:\n        frame[name] = average_available([past(target, slot - d * 48) for d in days]).ravel()\n    profile = np.zeros((len(origins), len(routes)))\n    counts = np.zeros_like(profile)\n    for day in range(1, 29):\n        level = past(level24, origins - day * 48)\n        adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)\n        profile += np.nan_to_num(adjusted)\n        counts += np.isfinite(adjusted)\n    frame['profile28'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()\n    if h == 1:\n        lower = np.zeros_like(target[origins])\n        upper = past(target, origins - 2)\n        for position in (origins - 1, origins):\n            total = past(target, position)\n            lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))\n        seasonal = frame[['slot_daily7', 'slot_weekly4']].mean(axis=1).to_numpy()\n        frame['overlap'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)\n    else:\n        frame['overlap'] = np.nan\n    numeric = frame.columns.difference(['cutoff', 'timestamp', 'route_id'])\n    return frame.astype({name: 'float32' for name in numeric})"




AFFINE_SLOPE_BOUNDS = (0.5, 1.5)
PARAMETER_SOURCES = {'1': {'path': 'notebooks/xgb_tuning/best_params/best_h1.json', 'sha256': '414fdb301f74cf1805091c599bc2b4344120e3ad357771bd26f5af3ca5c3002c'}, '2': {'path': 'notebooks/xgb_tuning/best_params/best_h2.json', 'sha256': '5b0bda47449c508f100bd32716c4c2781e44ccd8ea46370b099a7eed0ed54913'}, '3': {'path': 'notebooks/xgb_tuning/best_params/best_h3.json', 'sha256': 'ef082ae41cebdcba91c52bb7e7d7801ce18d3bae0810bcc3451027439076808f'}, '4': {'path': 'notebooks/xgb_tuning/best_params/best_h4.json', 'sha256': 'ea42f92b3bbb3f77e8de5e05444490386a3e4581b0b4c323ad304e4afee41fee'}, '5': {'path': 'notebooks/xgb_tuning/best_params/best_h5.json', 'sha256': '67b93d450c6f057a98a123bf823a66efd9fbd6895d54db4cba545c7687d2e9ca'}, '6': {'path': 'notebooks/xgb_tuning/best_params/best_h6.json', 'sha256': '4ef58ab410cb91a5d3364bdfe1cc270f4056d1d5ee64be12aea6e301d1479383'}, '7': {'path': 'notebooks/xgb_tuning/best_params/best_h7.json', 'sha256': '41a1f68f573f85e75be9782029b7e5d6e961b55a8b2be1f94ea4f7f67794b066'}, '8': {'path': 'notebooks/xgb_tuning/best_params/best_h8.json', 'sha256': '9e381a57163175348eaa218849faba9637c20f0d0423b21349d28a8aeffead6c'}}



MODEL_CLUSTERS = (1, 8)
GPU_BATCH_ROWS = 65536
GPU_MAX_BIN = 256
GPU_QUANTILE_BATCHES = 8


CORRECTION_MODE = 'scalar'
META_MIN_PRED_SCALE = 0.01
META_STRENGTHS = (0.5, 1.0)
META_CONFIGS = [dict(max_depth=1, trees=50, learning_rate=0.03, reg_lambda=100., min_child_weight=100.), dict(max_depth=1, trees=100, learning_rate=0.03, reg_lambda=100., min_child_weight=100.), dict(max_depth=2, trees=150, learning_rate=0.03, reg_lambda=100., min_child_weight=100.)]
CODE_HASH = '7716dda632391ad9105952a880be3c176496c20007f9500fbfd16ca0c81129f5'


In [ ]:
from pathlib import Path
import gc
import hashlib
import importlib.metadata
import json
import time
import zipfile
import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from catboost.utils import get_gpu_device_count
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits
from IPython.display import display


def require_gpu():
    global cp
    if TASK_TYPE != 'GPU':
        raise ValueError("Обучение разрешено только на GPU: TASK_TYPE='GPU'")
    if get_gpu_device_count() < 1:
        raise RuntimeError('CatBoost не видит GPU. Включите NVIDIA GPU в настройках Kaggle.')
    try:
        import cupy as cp
        probe = cp.linalg.solve(cp.eye(2, dtype=cp.float64), cp.ones(2, dtype=cp.float64))
        if not np.allclose(cp.asnumpy(probe), 1):
            raise RuntimeError('Ошибка проверки CUDA')
        cp.cuda.get_current_stream().synchronize()
    except Exception as error:
        raise RuntimeError('Для Ridge требуется рабочий CuPy под CUDA вашей Kaggle-сессии. CPU fallback отключён.') from error
    print('GPU обязателен: CatBoost CUDA и CuPy проверены.', flush=True)


class GPURidge:
    def __init__(self, alpha):
        self.alpha = float(alpha)

    def fit(self, x, y, sample_weight=None):
        width = x.shape[1] + 1
        gram = cp.zeros((width, width), dtype=cp.float64)
        rhs = cp.zeros(width, dtype=cp.float64)
        for start in range(0, len(y), PREDICT_BATCH):
            stop = min(start + PREDICT_BATCH, len(y))
            z = cp.empty((stop - start, width), dtype=cp.float64)
            z[:, :-1] = cp.asarray(x[start:stop])
            z[:, -1] = 1
            weights = cp.ones(stop - start) if sample_weight is None else cp.asarray(sample_weight[start:stop])
            gram += z.T @ (z * weights[:, None])
            rhs += z.T @ (cp.asarray(y[start:stop]) * weights)
        penalty = cp.full(width, self.alpha, dtype=cp.float64)
        penalty[-1] = 0
        solution = cp.asnumpy(cp.linalg.solve(gram + cp.diag(penalty), rhs))
        if not np.isfinite(solution).all():
            raise ValueError('Неконечные коэффициенты GPU Ridge')
        self.coef_, self.intercept_ = solution[:-1], float(solution[-1])
        return self

    def predict(self, x):
        return cp.asnumpy(cp.asarray(x, dtype=cp.float64) @ cp.asarray(self.coef_) + self.intercept_)


def write_json(path, value):
    path = Path(path)
    temp = path.with_suffix('.tmp')
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False))
    temp.replace(path)


def digest_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024**2), b''):
            digest.update(block)
    return digest.hexdigest()


def load_inputs():
    global TIMES, ROUTES, VALUES, REQUESTS, VAL_START, TEST_START, TEST_END, RUN, ROLL
    require_gpu()
    if THREADS < 1 or PREDICT_BATCH < 1 or MIN_CLUSTER_ROUTES < 1:
        raise ValueError('Некорректные настройки исполнения')
    if not RIDGE_LAMBDAS or min(RIDGE_LAMBDAS) <= 0 or not CAT_CONFIGS:
        raise ValueError('Нужны положительные регуляризации и конфигурации CatBoost')
    train_path, test_path = Path(TRAIN_PATH), Path(TEST_PATH)
    if not (train_path.is_file() and test_path.is_file()) and Path('/kaggle/input').exists():
        matches = [list(Path('/kaggle/input').rglob(p.name)) for p in (train_path, test_path)]
        if any(len(m) != 1 for m in matches):
            raise ValueError('Укажите однозначные TRAIN_PATH и TEST_PATH')
        train_path, test_path = (m[0] for m in matches)
    history = pd.read_parquet(train_path, columns=['route_id', 'timestamp', TARGET, *STATUSES])
    REQUESTS = pd.read_parquet(test_path, columns=['id', 'route_id', 'timestamp'])
    for frame in (history, REQUESTS):
        frame['timestamp'] = pd.to_datetime(frame.timestamp)
        if frame.isna().any().any() or frame.duplicated(['route_id', 'timestamp']).any():
            raise ValueError('Пропуски или повторяющиеся пары маршрут × время')
    history = history.sort_values(['timestamp', 'route_id'])
    TIMES = pd.DatetimeIndex(history.timestamp.unique())
    ROUTES = np.sort(history.route_id.unique())
    if len(history) != len(TIMES) * len(ROUTES) or not TIMES.equals(pd.date_range(TIMES[0], TIMES[-1], freq='30min')):
        raise ValueError('Нужна полная получасовая панель')
    if (TIMES[-1].hour, TIMES[-1].minute) != (10, 30):
        raise ValueError('Ожидался последний cutoff в 10:30')
    expected = pd.MultiIndex.from_product([ROUTES, [TIMES[-1] + h * STEP for h in HORIZONS]])
    actual = pd.MultiIndex.from_frame(REQUESTS[['route_id', 'timestamp']])
    if not REQUESTS.id.is_unique or len(actual) != len(expected) or len(expected.difference(actual)):
        raise ValueError('Конкурсный test не соответствует маршрутам и горизонтам')
    VALUES = {c: history[c].to_numpy(dtype=np.float32).reshape(len(TIMES), len(ROUTES)) for c in [TARGET, *STATUSES]}
    del history
    if any(not np.isfinite(a).all() for a in VALUES.values()) or (VALUES[TARGET] < 0).any():
        raise ValueError('Некорректные значения истории')
    full_dates = TIMES[(TIMES.hour == 10) & (TIMES.minute == 30) & (TIMES + max(HORIZONS) * STEP <= TIMES[-1])]
    if len(full_dates) <= VAL_DAYS + TEST_DAYS + 28:
        raise ValueError('Недостаточно истории для train/val/test')
    TEST_START = full_dates[-TEST_DAYS].normalize()
    VAL_START = full_dates[-TEST_DAYS - VAL_DAYS].normalize()
    TEST_END = full_dates[-1].normalize() + pd.Timedelta(days=1)
    if min(CLUSTERS) != 1 or max(CLUSTERS) > len(ROUTES):
        raise ValueError('CLUSTERS должен включать 1 и не превышать число маршрутов')
    config = dict(version=PIPELINE_VERSION, code=CODE_HASH, clusters=list(CLUSTERS), horizons=list(HORIZONS),
                  min_cluster_routes=MIN_CLUSTER_ROUTES, ridge_lambdas=list(RIDGE_LAMBDAS), cat_configs=CAT_CONFIGS,
                  seed=SEED, val_days=VAL_DAYS, test_days=TEST_DAYS, task_type=TASK_TYPE,
                  versions={p: importlib.metadata.version(p) for p in ('numpy', 'pandas', 'scikit-learn', 'catboost', 'pyarrow')},
                  train_hash=digest_file(train_path), test_hash=digest_file(test_path))
    run_id = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:16]
    RUN = Path(OUTPUT_ROOT) / ('cluster_experiment_' + run_id)
    RUN.mkdir(parents=True, exist_ok=True)
    write_json(RUN / 'manifest.json', config)
    splits = pd.DataFrame([
        dict(period='train', first=str(TIMES[0]), end_exclusive=str(VAL_START), starts='все 48'),
        dict(period='val', first=str(VAL_START), end_exclusive=str(TEST_START), starts='10:30'),
        dict(period='test', first=str(TEST_START), end_exclusive=str(TEST_END), starts='10:30'),
    ])
    splits.to_csv(RUN / 'splits.csv', index=False)
    display(splits)
    ROLL = {w: pd.DataFrame(VALUES[TARGET]).rolling(w, min_periods=1).mean().to_numpy(dtype=np.float32) for w in (4, 12, 48, 336)}
    print('Маршрутов:', len(ROUTES), 'Выходы:', RUN, 'CatBoost:', TASK_TYPE)


def segment_routes(boundary, stage):
    mask = TIMES < boundary
    y = VALUES[TARGET][mask].astype(np.float64)
    scale = np.maximum(y.mean(axis=0), 1.0)
    normalized = y / scale
    grid = TIMES[mask]
    descriptors = {}
    for slot in range(48):
        select = grid.hour * 2 + grid.minute // 30 == slot
        descriptors[f'slot_{slot:02}'] = normalized[select].mean(axis=0)
    for day in range(7):
        descriptors[f'weekday_{day}'] = normalized[grid.dayofweek == day].mean(axis=0)
    descriptors['log_cv'] = np.log1p(normalized.std(axis=0))
    descriptors['zero_share'] = (y == 0).mean(axis=0)
    descriptors['jump_share'] = (np.abs(np.diff(normalized, axis=0)) > 1).mean(axis=0)
    descriptors['log_q95'] = np.log1p(np.quantile(normalized, 0.95, axis=0))
    frame = pd.DataFrame(descriptors, index=ROUTES)
    if not np.isfinite(frame.to_numpy()).all():
        raise ValueError('Недостаточно данных для профилей маршрутов')
    scaler = StandardScaler().fit(frame)
    z = scaler.transform(frame)
    labels, estimators = {1: np.zeros(len(ROUTES), dtype=int)}, {}
    for k in CLUSTERS:
        if k == 1:
            continue
        model = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit(z)
        groups = model.labels_.copy()
        counts = np.bincount(groups, minlength=k)
        groups[counts[groups] < MIN_CLUSTER_ROUTES] = -1
        labels[k], estimators[k] = groups, model
    artifact = dict(boundary=str(boundary), routes=ROUTES, scale=scale, labels=labels,
                    descriptor_scaler=scaler, clusterers=estimators, descriptor_columns=list(frame.columns))
    joblib.dump(artifact, RUN / f'segments_{stage}.joblib')
    frame.insert(0, 'route_id', ROUTES)
    frame['target_scale'] = scale
    for k, groups in labels.items():
        frame[f'cluster_{k}'] = groups
    frame.to_csv(RUN / f'route_profiles_{stage}.csv', index=False)
    display(pd.DataFrame([dict(stage=stage, k=k, cluster=int(g), routes=int((groups == g).sum()))
                          for k, groups in labels.items() for g in np.unique(groups)]))
    return artifact


def origin_indices(h, period):
    labelled = np.arange(len(TIMES) - h)
    targets = TIMES[labelled + h]
    morning = (TIMES[labelled].hour == 10) & (TIMES[labelled].minute == 30)
    masks = dict(train=targets < VAL_START, train_val=targets < TEST_START,
                 val=(TIMES[labelled] >= VAL_START) & (targets < TEST_START) & morning,
                 test=(TIMES[labelled] >= TEST_START) & (TIMES[labelled] < TEST_END) & morning,
                 full=np.ones(len(labelled), dtype=bool))
    return labelled[masks[period]]


def features(h, origins, segmentation):
    scale = segmentation['scale'].astype(np.float32)
    columns, names = [], []
    shape = (len(origins), len(ROUTES))
    def add(name, array):
        names.append(name)
        columns.append(np.broadcast_to(array, shape).reshape(-1))
    def lag(name, array, indices):
        valid = indices >= 0
        data = array[np.maximum(indices, 0)] / scale
        add(name, np.where(valid[:, None], data, 0))
        add(name + '_available', valid[:, None])
    for delay in (0, 1, 2, 4, 12):
        lag(f'target_back_{delay}', VALUES[TARGET], origins - delay)
    for status in STATUSES:
        lag(status, VALUES[status], origins)
        add(status + '_delta_1h', (VALUES[status][origins] - VALUES[status][np.maximum(origins - 2, 0)]) / scale)
    for window, array in ROLL.items():
        lag(f'target_mean_{window}', array, origins)
    for days in (1, 7, 14, 28):
        lag(f'target_season_{days}', VALUES[TARGET], origins + h - 48 * days)
    dates = TIMES[origins] + h * STEP
    for label, phase in [('weekday', dates.dayofweek.to_numpy() / 7), ('slot', (dates.hour * 2 + dates.minute / 30).to_numpy() / 48)]:
        add(label + '_sin', np.sin(2 * np.pi * phase)[:, None])
        add(label + '_cos', np.cos(2 * np.pi * phase)[:, None])
    add('log_route_scale', np.log1p(scale)[None, :])
    matrix = np.column_stack(columns).astype(np.float32)
    if not np.isfinite(matrix).all():
        raise ValueError('Неконечные признаки')
    return matrix, names


def dataset(h, period, segmentation):
    origins = origin_indices(h, period)
    x, names = features(h, origins, segmentation)
    y = VALUES[TARGET][origins + h].reshape(-1).astype(np.float64)
    route_index = np.tile(np.arange(len(ROUTES)), len(origins))
    return dict(x=x, y=y, route_index=route_index, origins=origins, names=names)


def metrics(y, pred):
    if not np.isfinite(pred).all():
        raise ValueError('Неконечный прогноз')
    errors = pred - y
    total = float(y.sum())
    if total <= 0:
        raise ValueError('WAPE не определена при нулевом суммарном факте')
    absolute, signed = float(np.abs(errors).sum()), float(errors.sum())
    return dict(n=len(y), volume=total, absolute=absolute, signed=signed,
                WAPE=absolute / total, relative_bias=signed / total,
                score=(absolute + abs(signed)) / total)


def aggregate(rows, keys):
    table = pd.DataFrame(rows)
    totals = table.groupby(keys, as_index=False)[['n', 'volume', 'absolute', 'signed']].sum()
    totals['WAPE'] = totals.absolute / totals.volume
    totals['abs_relative_bias'] = totals.signed.abs() / totals.volume
    totals['WAPE + |Relative Bias|'] = totals.WAPE + totals.abs_relative_bias
    return totals


def fit_model(train, segmentation, family, k, settings):
    scales = segmentation['scale'][train['route_index']]
    target = train['y'] / scales
    weights = scales / scales.mean()
    labels = segmentation['labels'][k][train['route_index']]
    models = {}
    groups = np.unique(labels)
    for group in groups:
        select = np.ones(len(labels), dtype=bool) if group == -1 else labels == group
        x, y, w = train['x'][select], target[select], weights[select]
        started = time.monotonic()
        if family == 'ridge':
            scaler = StandardScaler().fit(x)
            z = scaler.transform(x)
            model = GPURidge(alpha=float(settings['lambda']) * len(x))
            model.fit(z, y, sample_weight=w)
        else:
            scaler = None
            model = CatBoostRegressor(**settings, loss_function='MAE', random_seed=SEED,
                                      task_type=TASK_TYPE, thread_count=THREADS, verbose=False,
                                      allow_writing_files=False, allow_const_label=True, l2_leaf_reg=10)
            model.fit(x, y, sample_weight=w)
        models[int(group)] = dict(model=model, scaler=scaler)
        print(f'{family} k={k} group={group}: {len(y):,} строк, {time.monotonic() - started:.1f} с', flush=True)
        del x, y, w
    return dict(models=models, family=family, k=k, settings=settings, feature_names=train['names'])


def predict_model(bundle, data, segmentation):
    labels = segmentation['labels'][bundle['k']][data['route_index']]
    pred = np.empty(len(labels), dtype=np.float64)
    for group, item in bundle['models'].items():
        indices = np.flatnonzero(labels == group)
        for start in range(0, len(indices), PREDICT_BATCH):
            rows = indices[start:start + PREDICT_BATCH]
            x = data['x'][rows]
            if item['scaler'] is not None:
                x = item['scaler'].transform(x)
            pred[rows] = item['model'].predict(x)
    pred *= segmentation['scale'][data['route_index']]
    return np.maximum(pred, 0)


def variants():
    return [('ridge_global', 'ridge', 1), *[(f'cat_k{k}', 'cat', k) for k in CLUSTERS]]


def run_search():
    segmentation = segment_routes(VAL_START, 'search')
    rows, selected = [], {}
    for h in HORIZONS:
        train, val = dataset(h, 'train', segmentation), dataset(h, 'val', segmentation)
        for name, family, k in variants():
            choices = [dict(lambda_=float(v)) for v in RIDGE_LAMBDAS] if family == 'ridge' else CAT_CONFIGS
            choices = [{'lambda': c['lambda_']} if family == 'ridge' else dict(c) for c in choices]
            path = RUN / f'search_h{h}_{name}.json'
            if path.exists():
                saved = json.loads(path.read_text())
                selected[f'{name}_h{h}'] = saved['best']
                rows.extend(saved['rows'])
                continue
            local, best = [], None
            for candidate, settings in enumerate(choices):
                bundle = fit_model(train, segmentation, family, k, settings)
                report = []
                for period, data in [('train', train), ('val', val)]:
                    record = dict(variant=name, h=h, candidate=candidate, period=period,
                                  **metrics(data['y'], predict_model(bundle, data, segmentation)))
                    local.append(record)
                    report.append(record)
                display(pd.DataFrame(report)[['variant', 'h', 'candidate', 'period', 'WAPE', 'relative_bias', 'score']])
                print('Test закрыт до фиксации настроек; score = WAPE + |Relative Bias|', flush=True)
                score = report[-1]['score']
                if best is None or score < best['score']:
                    best = dict(candidate=candidate, settings=settings, score=score)
                del bundle
                gc.collect()
            write_json(path, dict(best=best, rows=local))
            selected[f'{name}_h{h}'] = best
            rows.extend(local)
        del train, val
        gc.collect()
    chosen = [r for r in rows if r['candidate'] == selected[f"{r['variant']}_h{r['h']}"]['candidate']]
    summary = aggregate(chosen, ['variant', 'period'])
    validation = summary[summary.period == 'val'].sort_values(['WAPE + |Relative Bias|', 'variant'])
    winner = str(validation.iloc[0].variant)
    frozen = dict(selected=selected, winner=winner, selection_period='val', val_start=str(VAL_START), test_start=str(TEST_START))
    write_json(RUN / 'selection_frozen.json', frozen)
    pd.DataFrame(rows).to_csv(RUN / 'search_metrics.csv', index=False)
    summary.to_csv(RUN / 'validation_summary.csv', index=False)
    display(summary)
    print('Выбран по val:', winner, flush=True)
    return frozen


def save_predictions(h, name, period, data, pred):
    origins = data['origins']
    frame = pd.DataFrame(dict(route_id=ROUTES[data['route_index']], cutoff=np.repeat(TIMES[origins], len(ROUTES)),
                              timestamp=np.repeat(TIMES[origins] + h * STEP, len(ROUTES)), h=h, y=data['y'], y_pred=pred))
    frame.to_parquet(RUN / f'{period}_h{h}_{name}.parquet', index=False)


def run_evaluation(frozen):
    segmentation = segment_routes(TEST_START, 'evaluation')
    rows = []
    for h in HORIZONS:
        train, test = dataset(h, 'train_val', segmentation), dataset(h, 'test', segmentation)
        for name, family, k in variants():
            path = RUN / f'evaluation_h{h}_{name}.json'
            if path.exists():
                rows.extend(json.loads(path.read_text()))
                continue
            settings = frozen['selected'][f'{name}_h{h}']['settings']
            bundle = fit_model(train, segmentation, family, k, settings)
            report = []
            for period, data in [('train', train), ('test', test)]:
                pred = predict_model(bundle, data, segmentation)
                report.append(dict(variant=name, h=h, period=period, **metrics(data['y'], pred)))
                if period == 'test':
                    save_predictions(h, name, period, data, pred)
            display(pd.DataFrame(report)[['variant', 'h', 'period', 'WAPE', 'relative_bias', 'score']])
            write_json(path, report)
            rows.extend(report)
            del bundle
            gc.collect()
        del train, test
        gc.collect()
    summary = aggregate(rows, ['variant', 'period'])
    summary['selected_on_val'] = summary.variant == frozen['winner']
    pd.DataFrame(rows).to_csv(RUN / 'independent_metrics_by_horizon.csv', index=False)
    summary.to_csv(RUN / 'independent_metrics.csv', index=False)
    display(summary)
    note = 'Независимая оценка test получена до переобучения на всей доступной размеченной истории. Train здесь включает бывшую val. Период test уже использовался в других экспериментах проекта. Победитель выбран только по val.'
    (RUN / 'evaluation_note.txt').write_text(note)
    print(note)
    write_json(RUN / 'evaluation_complete.json', dict(winner=frozen['winner']))
    return summary


def run_full(frozen):
    if not (RUN / 'evaluation_complete.json').exists():
        raise ValueError('Сначала завершите независимую оценку')
    segmentation = segment_routes(TIMES[-1] + STEP, 'full')
    name, family, k = next(v for v in variants() if v[0] == frozen['winner'])
    rows, predictions = [], []
    for h in HORIZONS:
        model_path = RUN / f'final_model_h{h}.joblib'
        train = dataset(h, 'full', segmentation)
        settings = frozen['selected'][f'{name}_h{h}']['settings']
        if model_path.exists():
            bundle = joblib.load(model_path)
        else:
            bundle = fit_model(train, segmentation, family, k, settings)
            temporary = model_path.with_suffix('.tmp')
            joblib.dump(bundle, temporary, compress=3)
            temporary.replace(model_path)
        rows.append(dict(variant=name, h=h, period='full_in_sample',
                         **metrics(train['y'], predict_model(bundle, train, segmentation))))
        x, feature_names = features(h, np.array([len(TIMES) - 1]), segmentation)
        if feature_names != bundle['feature_names']:
            raise ValueError('Изменился порядок признаков')
        data = dict(x=x, route_index=np.arange(len(ROUTES)))
        pred = predict_model(bundle, data, segmentation)
        predictions.append(pd.DataFrame(dict(route_id=ROUTES, timestamp=TIMES[-1] + h * STEP, y_pred=pred)))
        independent = pd.read_csv(RUN / 'independent_metrics_by_horizon.csv')
        display(pd.concat([independent[(independent.variant == name) & (independent.h == h)], pd.DataFrame([rows[-1]])])[
            ['variant', 'h', 'period', 'WAPE', 'relative_bias', 'score']])
        print('Строки train/test относятся к модели ДО полного переобучения.', flush=True)
        del train, bundle
        gc.collect()
    forecast = pd.concat(predictions, ignore_index=True)
    submission = REQUESTS.merge(forecast, on=['route_id', 'timestamp'], how='left', validate='one_to_one', sort=False)
    if len(submission) != len(REQUESTS) or not submission.id.equals(REQUESTS.id) or not np.isfinite(submission.y_pred).all() or (submission.y_pred < 0).any():
        raise ValueError('Нарушен формат сабмита')
    submission[['id', 'y_pred']].to_csv(RUN / 'submission.csv', index=False)
    aggregate(rows, ['variant', 'period']).to_csv(RUN / 'full_in_sample_metrics.csv', index=False)
    if CREATE_ARCHIVE:
        with zipfile.ZipFile(RUN.with_suffix('.zip'), 'w', compression=zipfile.ZIP_DEFLATED) as archive:
            for path in sorted(RUN.rglob('*')):
                if path.is_file():
                    archive.write(path, path.relative_to(RUN.parent))
    print('Сабмит:', RUN / 'submission.csv')
    print('Независимая оценка test получена до переобучения на всей доступной размеченной истории.')

In [ ]:
display(pd.DataFrame([dict(h=int(h), trees=p['trees'], **p['params']) for h, p in PARAMETERS[EXPERIMENTS[0]['parameter_set']].items()]))


In [ ]:
def calibration_variants():
    return [(f'cat_k{k}', k) for k in CLUSTERS]


def initialize_calibration():
    global CAL_SPLIT, CAL_GROUP_LABELS
    if not 0 < CAL_FIT_DAYS < VAL_DAYS or not 0 < SCALE_BOUNDS[0] <= 1 <= SCALE_BOUNDS[1]:
        raise ValueError('Некорректные периоды или границы калибровки')
    load_inputs()
    CAL_SPLIT = VAL_START + pd.Timedelta(days=CAL_FIT_DAYS)
    group_segmentation = segment_routes(VAL_START, 'calibration_groups')
    CAL_GROUP_LABELS = group_segmentation['labels'][max(CLUSTERS)].copy()
    pd.DataFrame(dict(route_id=ROUTES, calibration_group=CAL_GROUP_LABELS)).to_csv(RUN / 'calibration_groups.csv', index=False)
    display(pd.DataFrame([
        dict(period='calibration_fit', start=str(VAL_START), end_exclusive=str(CAL_SPLIT)),
        dict(period='calibration_select', start=str(CAL_SPLIT), end_exclusive=str(TEST_START)),
        dict(period='test', start=str(TEST_START), end_exclusive=str(TEST_END)),
    ]))


def calibration_data(h, origins, segmentation):
    x, names = features(h, origins, segmentation)
    return dict(x=x, names=names, origins=origins,
                y=VALUES[TARGET][origins + h].reshape(-1).astype(np.float64),
                route_index=np.tile(np.arange(len(ROUTES)), len(origins)))


def prediction_frame(h, data, pred):
    return pd.DataFrame(dict(route_id=ROUTES[data['route_index']],
                             cutoff=np.repeat(TIMES[data['origins']], len(ROUTES)),
                             timestamp=np.repeat(TIMES[data['origins']] + h * STEP, len(ROUTES)),
                             h=h, calibration_group=CAL_GROUP_LABELS[data['route_index']], y=data['y'], y_pred=pred))


def scalar_ratio(frame):
    actual, predicted = float(frame.y.sum()), float(frame.y_pred.sum())
    if actual <= 0 or predicted <= 0 or not np.isfinite(actual + predicted):
        raise ValueError('Для калибровки нужны положительные конечные суммы факта и прогноза')
    return actual / predicted


def fit_factors(frame, strategy):
    if set(frame.h.unique()) != set(HORIZONS):
        raise ValueError('Калибровочная выборка должна содержать все горизонты')
    if strategy == 'raw':
        return {str(h): 1.0 for h in HORIZONS}
    if strategy in ('group', 'group_shrink50', 'group_horizon', 'group_horizon_shrink50'):
        if 'calibration_group' not in frame:
            raise ValueError('Нет фиксированных групп калибровки')
        by_horizon = strategy.startswith('group_horizon')
        baseline = fit_factors(frame, 'horizon' if by_horizon else 'global')
        factors = dict(baseline, groups={})
        shrink = 0.5 if strategy.endswith('shrink50') else 1.0
        for group, part in frame.groupby('calibration_group'):
            if group < 0 or part.route_id.nunique() < MIN_CLUSTER_ROUTES:
                continue
            values = {}
            for h in HORIZONS:
                local = part[part.h == h] if by_horizon else part
                if local.empty or local.y.sum() <= 0 or local.y_pred.sum() <= 0:
                    continue
                ratio = scalar_ratio(local)
                values[str(h)] = float(np.clip((1 - shrink) * baseline[str(h)] + shrink * ratio, *SCALE_BOUNDS))
            if values:
                factors['groups'][str(int(group))] = values
        return factors
    global_ratio = scalar_ratio(frame)
    if strategy not in ('global', 'horizon_shrink50', 'horizon'):
        raise ValueError('Неизвестный способ калибровки')
    shrink = {'global': 0.0, 'horizon_shrink50': 0.5, 'horizon': 1.0}[strategy]
    factors = {}
    for h in HORIZONS:
        local_ratio = scalar_ratio(frame[frame.h == h])
        value = (1 - shrink) * global_ratio + shrink * local_ratio
        factors[str(h)] = float(np.clip(value, *SCALE_BOUNDS))
    return factors


def corrected(frame, factors):
    multiplier = frame.h.map(lambda h: factors[str(int(h))]).to_numpy(dtype=np.float64, copy=True)
    for group, values in factors.get('groups', {}).items():
        for h, value in values.items():
            selected = (frame.calibration_group.to_numpy() == int(group)) & (frame.h.to_numpy() == int(h))
            multiplier[selected] = value
    pred = frame.y_pred.to_numpy(dtype=np.float64) * multiplier
    if not np.isfinite(pred).all() or (pred < 0).any():
        raise ValueError('Некорректный скорректированный прогноз')
    return pred


def apply_factors(raw, h, route_index, factors):
    per_route = np.full(len(ROUTES), factors[str(h)], dtype=np.float64)
    for group, values in factors.get('groups', {}).items():
        per_route[CAL_GROUP_LABELS == int(group)] = values.get(str(h), factors[str(h)])
    return raw * per_route[route_index]


def factor_records(factors):
    rows = [dict(scope='fallback', calibration_group=-1, h=h, factor=factors[str(h)]) for h in HORIZONS]
    for group, values in factors.get('groups', {}).items():
        rows.extend(dict(scope='group', calibration_group=int(group), h=int(h), factor=value) for h, value in values.items())
    return rows


def phase_forecasts(stage, boundary, query_start, query_end, frozen=None):
    if stage == 'test' and (frozen is None or not (RUN / 'calibration_frozen.json').exists()):
        raise ValueError('Test закрыт до фиксации калибровки')
    segmentation = segment_routes(boundary, stage)
    all_rows = []
    for h in HORIZONS:
        positions = np.arange(len(TIMES) - h)
        train_origins = positions[TIMES[positions + h] < boundary]
        query_mask = ((TIMES[positions] >= query_start) & (TIMES[positions] < query_end)
                      & (TIMES[positions].hour == 10) & (TIMES[positions].minute == 30)
                      & (TIMES[positions + h] < query_end))
        query_origins = positions[query_mask]
        if not len(train_origins) or not len(query_origins):
            raise ValueError('Пустой обучающий или проверочный период')
        train = calibration_data(h, train_origins, segmentation)
        query = calibration_data(h, query_origins, segmentation)
        for name, k in calibration_variants():
            path = RUN / f'{stage}_h{h}_{name}.parquet'
            metric_path = RUN / f'{stage}_h{h}_{name}_metrics.json'
            if path.exists() and metric_path.exists():
                all_rows.extend(json.loads(metric_path.read_text()))
                continue
            bundle = fit_model(train, segmentation, 'cat', k, CAT_CONFIGS[0])
            report = []
            for period, data in [('train', train), (stage, query)]:
                raw = predict_model(bundle, data, segmentation)
                report.append(dict(variant=name, h=h, strategy='raw', period=period, **metrics(data['y'], raw)))
                if frozen is not None:
                    choice = frozen['variants'][name]
                    pred = apply_factors(raw, h, data['route_index'], choice['factors'])
                    report.append(dict(variant=name, h=h, strategy='selected', period=period, **metrics(data['y'], pred)))
                    for strategy, candidate in frozen['comparison_factors'][name].items():
                        if strategy == 'raw':
                            continue
                        candidate_pred = apply_factors(raw, h, data['route_index'], candidate)
                        report.append(dict(variant=name, h=h, strategy=strategy, period=period, **metrics(data['y'], candidate_pred)))
                if period == stage:
                    frame = prediction_frame(h, data, raw)
                    if frozen is not None:
                        frame['y_calibrated'] = pred
                    frame.to_parquet(path, index=False)
            write_json(metric_path, report)
            all_rows.extend(report)
            display(pd.DataFrame(report)[['variant', 'h', 'strategy', 'period', 'WAPE', 'relative_bias', 'score']])
            if frozen is None:
                print('Test закрыт. Калибровка и выбор используют только периоды до test.', flush=True)
            del bundle
            gc.collect()
        del train, query
        gc.collect()
    summary = aggregate(all_rows, ['variant', 'strategy', 'period'])
    summary['relative_bias'] = summary.signed / summary.volume
    pd.DataFrame(all_rows).to_csv(RUN / f'{stage}_metrics_by_horizon.csv', index=False)
    summary.to_csv(RUN / f'{stage}_metrics.csv', index=False)
    display(summary)
    print(summary[['variant','strategy','period','WAPE','relative_bias','WAPE + |Relative Bias|']].to_string(index=False),flush=True)
    return summary


def read_phase(stage, name):
    frame = pd.concat([pd.read_parquet(RUN / f'{stage}_h{h}_{name}.parquet') for h in HORIZONS], ignore_index=True)
    if frame.duplicated(['route_id', 'cutoff', 'h']).any() or not np.isfinite(frame[['y', 'y_pred']]).all().all():
        raise ValueError('Некорректные сохранённые прогнозы')
    return frame


def select_calibration():
    rows, choices, factor_rows = [], {}, []
    strategies = ('raw', 'global', 'horizon_shrink50', 'horizon',
                  'group_shrink50', 'group', 'group_horizon_shrink50', 'group_horizon', 'recent_global_2', 'recent_global_3', 'recent_global_7')
    for name, k in calibration_variants():
        fit = read_phase('calibration_fit', name)
        select = read_phase('calibration_select', name)
        if not ((fit.cutoff >= VAL_START) & (fit.timestamp < CAL_SPLIT)).all():
            raise ValueError('Пересечение периода оценки коэффициентов с выбором')
        if not ((select.cutoff >= CAL_SPLIT) & (select.timestamp < TEST_START)).all():
            raise ValueError('Период выбора пересекает test')
        candidates = []
        for strategy in strategies:
            factors = fit_factors(fit, strategy)
            factor_rows.extend(dict(variant=name, strategy=strategy, fitted_on='calibration_fit', **row) for row in factor_records(factors))
            for period, frame in [('calibration_fit', fit), ('calibration_select', select)]:
                record = dict(variant=name, strategy=strategy, period=period,
                              **metrics(frame.y.to_numpy(), corrected(frame, factors)))
                rows.append(record)
                if period == 'calibration_select':
                    candidates.append(record)
        best = min(candidates, key=lambda row: row['score'])
        chosen_strategy = best['strategy']
        all_val = pd.concat([fit, select], ignore_index=True)
        factors = fit_factors(all_val, chosen_strategy)
        choices[name] = dict(k=k, strategy=chosen_strategy, selection_score=best['score'], factors=factors)
    winner = min(choices, key=lambda name: choices[name]['selection_score'])
    frozen = dict(winner=winner, variants=choices, base_settings=CAT_CONFIGS[0],
                  coefficient_fit_for_selection='first_val_block', selection_period='second_val_block',
                  evaluation_coefficient_fit='all_val_rolling_oof', test_used=False,
                  val_start=str(VAL_START), calibration_split=str(CAL_SPLIT), test_start=str(TEST_START),
                  calibration_groups=dict(k=max(CLUSTERS), fitted_before=str(VAL_START),
                                          routes=[str(r) for r in ROUTES], labels=CAL_GROUP_LABELS.tolist()))
    write_json(RUN / 'calibration_frozen.json', frozen)
    table = pd.DataFrame(rows)
    table.to_csv(RUN / 'calibration_selection.csv', index=False)
    pd.DataFrame(factor_rows).to_csv(RUN / 'calibration_candidates_factors.csv', index=False)
    display(table[['variant', 'strategy', 'period', 'WAPE', 'relative_bias', 'score']])
    display(pd.DataFrame([dict(variant=name, strategy=choice['strategy'], **row)
                          for name, choice in choices.items() for row in factor_records(choice['factors'])]))
    print('Победитель выбран до test:', winner, choices[winner]['strategy'], 'selection_score', choices[winner]['selection_score'], flush=True)
    return frozen


def evaluate_calibration(frozen):
    if frozen != json.loads((RUN / 'calibration_frozen.json').read_text()):
        raise ValueError('Настройки отличаются от зафиксированных до test')
    if frozen['calibration_groups']['labels'] != CAL_GROUP_LABELS.tolist() or frozen['calibration_groups']['routes'] != [str(r) for r in ROUTES]:
        raise ValueError('Изменились фиксированные группы калибровки')
    summary = phase_forecasts('test', TEST_START, TEST_START, TEST_END, frozen)
    summary['selected_on_val'] = (summary.variant == frozen['winner']) & (summary.strategy == 'selected')
    summary.to_csv(RUN / 'independent_calibration_metrics.csv', index=False)
    display(summary)
    print(summary[['variant','strategy','period','WAPE','relative_bias','WAPE + |Relative Bias|']].to_string(index=False),flush=True)
    note = 'Независимая оценка test получена до переобучения на всей доступной размеченной истории. Этот test ранее изучался в проекте и не является новым слепым holdout. Коэффициенты, способ калибровки и вариант модели зафиксированы до test. Train включает бывшую val.'
    (RUN / 'evaluation_note.txt').write_text(note)
    write_json(RUN / 'calibration_evaluation_complete.json', dict(frozen=frozen))
    print(note)
    return summary


def finalize_calibration(frozen):
    marker = RUN / 'calibration_evaluation_complete.json'
    if not marker.exists() or json.loads(marker.read_text())['frozen'] != frozen:
        raise ValueError('Сначала завершите оценку неизменённой процедуры')
    name = frozen['winner']
    choice = frozen['variants'][name]
    oof = pd.concat([read_phase(stage, name) for stage in ('calibration_fit', 'calibration_select', 'test')], ignore_index=True)
    final_factors = fit_factors(oof, choice['strategy'])
    write_json(RUN / 'final_calibration.json', dict(variant=name, strategy=choice['strategy'], factors=final_factors,
                                                   fit_period='available_rolling_oof_val_and_former_test',
                                                   independent_test_factors=choice['factors']))
    display(pd.DataFrame([dict(stage=stage, **row) for stage, factors in [('independent_test', choice['factors']), ('final', final_factors)]
                          for row in factor_records(factors)]))
    segmentation = segment_routes(TIMES[-1] + STEP, 'full')
    predictions, metric_rows = [], []
    for h in HORIZONS:
        train = calibration_data(h, np.arange(len(TIMES) - h), segmentation)
        model_path = RUN / f'calibrated_final_model_h{h}.joblib'
        if model_path.exists():
            bundle = joblib.load(model_path)
        else:
            bundle = fit_model(train, segmentation, 'cat', choice['k'], frozen['base_settings'])
            temporary = model_path.with_suffix('.tmp')
            joblib.dump(bundle, temporary, compress=3)
            temporary.replace(model_path)
        raw = predict_model(bundle, train, segmentation)
        for strategy, pred in [('raw', raw), ('selected', apply_factors(raw, h, train['route_index'], final_factors))]:
            metric_rows.append(dict(variant=name, h=h, strategy=strategy, period='full_in_sample', **metrics(train['y'], pred)))
        historical = pd.read_csv(RUN / 'test_metrics_by_horizon.csv')
        historical = historical[(historical.variant == name) & (historical.h == h)]
        display(pd.concat([historical, pd.DataFrame(metric_rows[-2:])], ignore_index=True)[
            ['variant', 'h', 'strategy', 'period', 'WAPE', 'relative_bias', 'score']])
        print('Train/test в таблице — прежние оценочные модели и прежние коэффициенты, до полного refit.', flush=True)
        x, names = features(h, np.array([len(TIMES) - 1]), segmentation)
        if names != bundle['feature_names']:
            raise ValueError('Изменился порядок признаков')
        raw_future = predict_model(bundle, dict(x=x, route_index=np.arange(len(ROUTES))), segmentation)
        predictions.append(pd.DataFrame(dict(route_id=ROUTES, timestamp=TIMES[-1] + h * STEP,
                                              y_pred=apply_factors(raw_future, h, np.arange(len(ROUTES)), final_factors), y_raw=raw_future)))
        del train, bundle
        gc.collect()
    forecast = pd.concat(predictions, ignore_index=True)
    submission = REQUESTS.merge(forecast, on=['route_id', 'timestamp'], how='left', sort=False, validate='one_to_one')
    if not submission.id.equals(REQUESTS.id) or not np.isfinite(submission[['y_pred', 'y_raw']]).all().all() or (submission[['y_pred', 'y_raw']] < 0).any().any():
        raise ValueError('Некорректный submission')
    submission[['id', 'y_pred']].to_csv(RUN / 'submission.csv', index=False)
    submission[['id', 'y_raw']].rename(columns={'y_raw': 'y_pred'}).to_csv(RUN / 'submission_raw.csv', index=False)
    aggregate(metric_rows, ['variant', 'strategy', 'period']).to_csv(RUN / 'full_calibration_metrics.csv', index=False)
    if CREATE_ARCHIVE:
        with zipfile.ZipFile(RUN.with_suffix('.zip'), 'w', compression=zipfile.ZIP_DEFLATED) as archive:
            for path in sorted(RUN.rglob('*')):
                if path.is_file():
                    archive.write(path, path.relative_to(RUN.parent))
    print('Сабмит:', RUN / 'submission.csv')
    print('Независимая оценка test получена до переобучения базовых моделей и обновления коэффициентов на всей доступной OOF-истории.')

In [ ]:
import xgboost as xgb
from datetime import datetime, timezone





def seasonal_base(x, names):
    total = np.zeros(len(x), dtype=np.float64)
    count = np.zeros(len(x), dtype=np.float64)
    for days in (7, 14, 28):
        col = names.index(f'target_season_{days}')
        available = x[:, names.index(f'target_season_{days}_available')]
        total += x[:, col] * available
        count += available
    return np.divide(total, count, out=x[:, names.index('target_back_0')].astype(np.float64).copy(), where=count > 0)


def fit_model(train, segmentation, family, k, settings):
    check_deadline()
    scales = segmentation['scale'][train['route_index']]
    target = train['y'] / scales
    if ACTIVE['residual']:
        target = target - seasonal_base(train['x'], train['names'])
    weights = scales / scales.mean()
    labels = segmentation['labels'][k][train['route_index']]
    params = dict(PARAMETERS[ACTIVE['parameter_set']][str(ACTIVE_H)]['params'])
    trees = PARAMETERS[ACTIVE['parameter_set']][str(ACTIVE_H)]['trees']
    if ACTIVE.get('compact'):
        trees = min(trees, 900)
        key = 'depth' if ACTIVE['algorithm'] == 'cat' else 'max_depth'
        params[key] = min(params[key], 6)
    models = {}
    for group in np.unique(labels):
        check_deadline()
        select = np.ones(len(labels), dtype=bool) if group == -1 else labels == group
        started = time.monotonic()
        if ACTIVE['algorithm'] == 'cat':
            model = CatBoostRegressor(**params, iterations=trees, loss_function=ACTIVE['loss'],
                                      task_type='GPU', devices='0', thread_count=THREADS,
                                      random_seed=SEED, verbose=False, allow_writing_files=False,
                                      allow_const_label=True, bootstrap_type='Bayesian')
        else:
            model = xgb.XGBRegressor(**params, n_estimators=trees, objective=ACTIVE['loss'],
                                     tree_method='hist', device='cuda:0', n_jobs=THREADS, random_state=SEED)
        model.fit(train['x'][select], target[select], sample_weight=weights[select])
        if ACTIVE['algorithm'] == 'xgb':
            device = json.loads(model.get_booster().save_config())['learner']['generic_param']['device']
            if not device.startswith('cuda'):
                raise RuntimeError('XGBoost переключился на CPU')
        models[int(group)] = dict(model=model, scaler=None)
        print(ACTIVE['name'], 'h', ACTIVE_H, 'k', k, 'group', int(group), 'seconds', round(time.monotonic()-started, 1), flush=True)
    return dict(models=models, k=k, residual=ACTIVE['residual'], feature_names=train['names'], experiment=ACTIVE)


def predict_model(bundle, data, segmentation):
    labels = segmentation['labels'][bundle['k']][data['route_index']]
    pred = np.empty(len(labels), dtype=np.float64)
    for group, item in bundle['models'].items():
        indices = np.flatnonzero(labels == group)
        for start in range(0, len(indices), PREDICT_BATCH):
            rows = indices[start:start + PREDICT_BATCH]
            pred[rows] = item['model'].predict(data['x'][rows])
    if bundle['residual']:
        pred += seasonal_base(data['x'], bundle['feature_names'])
    return np.maximum(pred * segmentation['scale'][data['route_index']], 0)


original_calibration_data = calibration_data


def calibration_data(h, origins, segmentation):
    global ACTIVE_H
    ACTIVE_H = h
    return original_calibration_data(h, origins, segmentation)





In [ ]:
original_fit_factors_recent = fit_factors


def fit_factors(frame, strategy):
    if strategy.startswith('recent_global_'):
        half_life = float(strategy.rsplit('_',1)[-1])
        age = (frame.cutoff.max() - frame.cutoff).dt.total_seconds().to_numpy() / 86400
        weight = np.exp2(-age / half_life)
        ratio = float((frame.y * weight).sum() / (frame.y_pred * weight).sum())
        if not np.isfinite(ratio) or ratio <= 0:
            raise ValueError('Некорректная взвешенная калибровка')
        return {str(h):float(np.clip(ratio,*SCALE_BOUNDS)) for h in HORIZONS}
    return original_fit_factors_recent(frame,strategy)


original_features_rich = features
original_segment_rich = segment_routes


def segment_routes(boundary, stage):
    global FEATURE_NS
    result = original_segment_rich(boundary, stage)
    FEATURE_NS = dict(np=np, pd=pd, values=VALUES, TARGET=TARGET, STATUSES=STATUSES,
                      times=TIMES, routes=ROUTES, STEP=STEP, WINDOW_OFFSETS=tuple(range(-4,5)))
    exec(RICH_SOURCE, FEATURE_NS)
    lag_table = FEATURE_NS['calculate_dynamic_shoulder'](VALUES[TARGET], FEATURE_NS['hub_signal'], TIMES, ROUTES, fit_before=boundary)
    FEATURE_NS['validation_lag_table'] = lag_table
    FEATURE_NS['network_values'] = {name:array.sum(axis=1,dtype=np.float64).astype(np.float32) for name,array in VALUES.items()}
    lag_table.to_csv(RUN / f'rich_status6_lags_{stage}.csv', index=False)
    return result


def features(h, origins, segmentation):
    base, names = original_features_rich(h, origins, segmentation)
    frame = FEATURE_NS['horizon_dataset'](h, origins)
    cols = [c for c in frame if c not in ('route_id','cutoff','timestamp','y','id','overlap')]
    values = frame[cols].to_numpy(dtype=np.float32)
    scale = np.tile(segmentation['scale'],len(origins)).astype(np.float32)
    dimensional = [i for i,c in enumerate(cols) if c not in ('hour','minute','hour_float','dayofweek','is_weekend') and not any(v in c for v in ('ratio','rate','correlation','pair_count','lag_steps','valid','available','_sin','_cos'))]
    for i in dimensional:
        values[:,i] /= scale
    available = np.isfinite(values).astype(np.float32)
    values = np.where(np.isfinite(values),values,0)
    shoulder_columns, shoulder_names = [], []
    for label, days in [('daily7',range(1,8)),('weekly4',(7,14,21,28))]:
        profiles = []
        for offset in (-4,-2,0,2,4):
            total = np.zeros((len(origins),len(ROUTES)),dtype=np.float32)
            count = np.zeros((len(origins),1),dtype=np.float32)
            for day in days:
                positions = origins + h - day*48 + offset
                valid = positions >= 0
                total += np.where(valid[:,None],VALUES[TARGET][np.maximum(positions,0)],0)
                count += valid[:,None]
            profile = np.divide(total,count,out=np.zeros_like(total),where=count>0) / segmentation['scale'][None,:]
            profiles.append(profile.ravel())
            shoulder_columns.append(profile.ravel())
            shoulder_names.append(f'rich_{label}_shoulder_{offset}')
        window = np.column_stack(profiles)
        shoulder_columns.extend([window.max(axis=1)-window.min(axis=1),np.argmax(window,axis=1)*60-120])
        shoulder_names.extend([f'rich_{label}_shoulder_range',f'rich_{label}_peak_minutes'])
    for signal, array in FEATURE_NS['network_values'].items():
        for lag in (0,2):
            signal_values = np.where(origins>=lag,array[np.maximum(origins-lag,0)],0) / max(float(segmentation['scale'].sum()),1.)
            shoulder_columns.append(np.repeat(signal_values,len(ROUTES)))
            shoulder_names.append(f'rich_network_{signal}_back_{lag}')
        change = (array[origins]-array[np.maximum(origins-2,0)]) / max(float(segmentation['scale'].sum()),1.)
        shoulder_columns.append(np.repeat(change,len(ROUTES)))
        shoulder_names.append(f'rich_network_{signal}_change_1h')
    shoulder_columns.append(np.tile(np.arange(len(ROUTES),dtype=np.float32),len(origins)))
    shoulder_names.append('rich_route_index')
    share = VALUES[TARGET][origins] / np.maximum(FEATURE_NS['network_values'][TARGET][origins,None],1)
    shoulder_columns.append(share.ravel())
    shoulder_names.append('rich_route_target_network_share')
    matrix = np.column_stack([base,values,available,*[np.asarray(c,dtype=np.float32) for c in shoulder_columns]]).astype(np.float32,copy=False)
    names = names + ['rich_'+c for c in cols] + ['rich_available__'+c for c in cols] + shoulder_names
    if not np.isfinite(matrix).all():
        raise ValueError('Неконечные расширенные признаки')
    return matrix,names


def model_frame(x,names,algorithm):
    frame=pd.DataFrame(x,columns=names)
    route_codes=x[:,names.index('rich_route_index')].astype(np.int64)
    frame['route_code']=pd.Categorical(route_codes,categories=np.arange(len(ROUTES))) if algorithm=='xgb' else route_codes.astype(str)
    return frame


def predict_model(bundle,data,segmentation):
    labels=segmentation['labels'][bundle['k']][data['route_index']]
    pred=np.empty(len(labels),dtype=np.float64)
    for group,item in bundle['models'].items():
        indices=np.flatnonzero(labels==group)
        for start in range(0,len(indices),PREDICT_BATCH):
            rows=indices[start:start+PREDICT_BATCH]
            frame=model_frame(data['x'][rows],bundle['feature_names'],bundle['experiment']['algorithm'])
            pred[rows]=item['model'].predict(frame)
    if bundle['residual']:
        pred+=seasonal_base(data['x'],bundle['feature_names'])
    return np.maximum(pred*segmentation['scale'][data['route_index']],0)


exec(RICH_TRAINING_SOURCE)


In [ ]:
from scipy.optimize import brentq, minimize_scalar

legacy_fit_factors = fit_factors
CALIBRATION_STRATEGIES = ('raw', 'fixed_1.025', 'fixed_1.05', 'global', 'horizon', 'horizon_shrink50',
                          'group', 'group_shrink50', 'group_horizon', 'group_horizon_shrink50',
                          'recent_global_2', 'recent_global_3', 'recent_global_7',
                          'offset', 'affine', 'affine_horizon', 'affine_group', 'affine_group_shrink50')


def affine_pair(frame, offset_only=False):
    scale = max(float(frame.y.mean()), 1.)
    x = frame.y_pred.to_numpy(dtype=np.float64) / scale
    y = frame.y.to_numpy(dtype=np.float64) / scale
    if not np.isfinite(x).all() or not np.isfinite(y).all() or y.sum() <= 0:
        raise ValueError('Некорректные данные affine-калибровки')

    def intercept(a):
        return float(brentq(lambda b: np.maximum(a * x + b, 0).sum() - y.sum(),
                            -max(float((a * x).max()), 1.), max(float(y.mean()), 1.), xtol=1e-10))

    def objective(a):
        b = intercept(a)
        error = np.maximum(a * x + b, 0) - y
        return float((np.abs(error).sum() + abs(error.sum())) / y.sum())

    if offset_only or np.ptp(x) < 1e-10:
        a = 1.
    else:
        search = minimize_scalar(objective, bounds=AFFINE_SLOPE_BOUNDS, method='bounded', options=dict(xatol=1e-5))
        candidates = [float(search.x), 1., *AFFINE_SLOPE_BOUNDS]
        a = min(candidates, key=objective)
    return float(a), intercept(a) * scale


def fit_factors(frame, strategy):
    if strategy.startswith('fixed_'):
        return {str(h): float(strategy.split('_', 1)[1]) for h in HORIZONS}
    if strategy not in ('offset', 'affine', 'affine_horizon', 'affine_group', 'affine_group_shrink50'):
        return legacy_fit_factors(frame, strategy)
    a, b = affine_pair(frame, strategy == 'offset')
    result = {str(h): a for h in HORIZONS}
    result['offsets'] = {str(h): b for h in HORIZONS}
    if strategy == 'affine_horizon':
        for h in HORIZONS:
            a, b = affine_pair(frame[frame.h == h])
            result[str(h)], result['offsets'][str(h)] = a, b
    if strategy.startswith('affine_group'):
        result['groups'], result['group_offsets'] = {}, {}
        strength = .5 if strategy.endswith('shrink50') else 1.
        for group, part in frame.groupby('calibration_group'):
            if group < 0 or part.route_id.nunique() < MIN_CLUSTER_ROUTES:
                continue
            a, b = affine_pair(part)
            key = str(int(group))
            result['groups'][key] = {str(h): (1 - strength) * result[str(h)] + strength * a for h in HORIZONS}
            result['group_offsets'][key] = {str(h): (1 - strength) * result['offsets'][str(h)] + strength * b for h in HORIZONS}
    return result


def calibration_coefficients(h, groups, factors):
    a = np.full(len(groups), factors[str(h)], dtype=np.float64)
    b = np.full(len(groups), factors.get('offsets', {}).get(str(h), 0.), dtype=np.float64)
    for group, values in factors.get('groups', {}).items():
        selected = groups == int(group)
        a[selected] = values.get(str(h), factors[str(h)])
        b[selected] = factors.get('group_offsets', {}).get(group, {}).get(str(h), factors.get('offsets', {}).get(str(h), 0.))
    return a, b


def corrected(frame, factors):
    output = np.empty(len(frame), dtype=np.float64)
    groups = frame.calibration_group.to_numpy()
    for h in HORIZONS:
        selected = frame.h.to_numpy() == h
        a, b = calibration_coefficients(h, groups[selected], factors)
        output[selected] = np.maximum(a * frame.y_pred.to_numpy()[selected] + b, 0)
    if not np.isfinite(output).all():
        raise ValueError('Неконечная калибровка')
    return output


def apply_factors(raw, h, route_index, factors):
    a, b = calibration_coefficients(h, CAL_GROUP_LABELS[route_index], factors)
    output = np.maximum(np.asarray(raw) * a + b, 0)
    if not np.isfinite(output).all():
        raise ValueError('Неконечная калибровка')
    return output


def factor_records(factors):
    groups = np.array([-1, *[int(g) for g in factors.get('groups', {})]])
    records = []
    for h in HORIZONS:
        a, b = calibration_coefficients(h, groups, factors)
        records.extend(dict(scope='fallback' if group == -1 else 'group', calibration_group=int(group), h=h,
                            factor=float(slope), offset=float(offset)) for group, slope, offset in zip(groups, a, b))
    return records


def select_calibration():
    rows, choices, factor_rows, comparisons = [], {}, [], {}
    for name, k in calibration_variants():
        fit, select = (read_phase(stage, name) for stage in ('calibration_fit', 'calibration_select'))
        if fit.timestamp.max() >= CAL_SPLIT or select.timestamp.max() >= TEST_START or select.cutoff.min() < CAL_SPLIT:
            raise ValueError('Пересечение периодов калибровки и test')
        candidates = []
        for strategy in CALIBRATION_STRATEGIES:
            factors = fit_factors(fit, strategy)
            factor_rows.extend(dict(variant=name, strategy=strategy, fitted_on='calibration_fit', **row) for row in factor_records(factors))
            for period, frame in [('calibration_fit', fit), ('calibration_select', select)]:
                row = dict(variant=name, strategy=strategy, period=period, **metrics(frame.y.to_numpy(), corrected(frame, factors)))
                rows.append(row)
                if period == 'calibration_select':
                    candidates.append(row)
        best = min(candidates, key=lambda row: row['score'])
        all_val = pd.concat([fit, select], ignore_index=True)
        comparisons[name] = {strategy: fit_factors(all_val, strategy) for strategy in CALIBRATION_STRATEGIES}
        choices[name] = dict(k=k, strategy=best['strategy'], selection_score=best['score'], factors=comparisons[name][best['strategy']])
    winner = min(choices, key=lambda name: choices[name]['selection_score'])
    frozen = dict(winner=winner, variants=choices, comparison_factors=comparisons, base_settings=CAT_CONFIGS[0],
                  coefficient_fit_for_selection='first_val_block', selection_period='second_val_block',
                  evaluation_coefficient_fit='all_val_rolling_oof', test_used=False,
                  val_start=str(VAL_START), calibration_split=str(CAL_SPLIT), test_start=str(TEST_START),
                  calibration_groups=dict(k=max(CLUSTERS), fitted_before=str(VAL_START), routes=[str(r) for r in ROUTES], labels=CAL_GROUP_LABELS.tolist()))
    write_json(RUN / 'calibration_frozen.json', frozen)
    pd.DataFrame(rows).to_csv(RUN / 'calibration_selection.csv', index=False)
    pd.DataFrame(factor_rows).to_csv(RUN / 'calibration_candidates_factors.csv', index=False)
    table = pd.DataFrame(rows)
    display(table[['variant', 'strategy', 'period', 'WAPE', 'relative_bias', 'score']])
    print(table[['variant', 'strategy', 'period', 'WAPE', 'relative_bias', 'score']].to_string(index=False), flush=True)
    print('Победитель зафиксирован до test:', winner, choices[winner]['strategy'], flush=True)
    return frozen


def check_deadline():
    return None


def require_gpu():
    if TASK_TYPE != 'GPU':
        raise ValueError('Обучение только на GPU')
    if ACTIVE['algorithm'] == 'cat' and get_gpu_device_count() < 1:
        raise RuntimeError('CatBoost не видит CUDA. Включите GPU в Kaggle.')
    if ACTIVE['algorithm'] == 'xgb':
        probe = xgb.XGBRegressor(n_estimators=1, max_depth=1, device='cuda:0', tree_method='hist', n_jobs=THREADS)
        probe.fit(np.arange(16, dtype=np.float32).reshape(8, 2), np.arange(8, dtype=np.float32))
        device = json.loads(probe.get_booster().save_config())['learner']['generic_param']['device']
        if not device.startswith('cuda'):
            raise RuntimeError('XGBoost не видит CUDA; CPU fallback запрещён')
        del probe
    print('GPU проверен:', ACTIVE['algorithm'], flush=True)


def run_rich_calibration():
    global ACTIVE, PIPELINE_VERSION, CAT_CONFIGS
    ACTIVE = EXPERIMENTS[0]
    PIPELINE_VERSION = 'rich_calibration_v3_' + ACTIVE['name']
    CAT_CONFIGS = [dict(experiment=ACTIVE, parameters=PARAMETERS, parameter_sources=PARAMETER_SOURCES)]
    STEP_VALUE = pd.Timedelta(minutes=30)
    if STEP != STEP_VALUE:
        raise ValueError('Неверный шаг времени')
    with threadpool_limits(limits=THREADS):
        initialize_calibration()
        write_json(RUN / 'calibration_protocol.json', dict(strategies=list(CALIBRATION_STRATEGIES), affine_slope_bounds=list(AFFINE_SLOPE_BOUNDS), test_protocol='all_coefficients_frozen_before_test', test_used_for_selection=False))
        pd.DataFrame([dict(h=int(h), trees=p['trees'], **p['params']) for h, p in PARAMETERS[ACTIVE['parameter_set']].items()]).to_csv(RUN / 'effective_model_params.csv', index=False)
        phase_forecasts('calibration_fit', VAL_START, VAL_START, CAL_SPLIT)
        phase_forecasts('calibration_select', CAL_SPLIT, CAL_SPLIT, TEST_START)
        frozen = select_calibration()
        comparison = evaluate_calibration(frozen)
        comparison.to_csv(RUN / 'all_calibrations_train_test.csv', index=False)
        print('Сравнение всех калибровок; победитель выбран до test:', flush=True)
        print(comparison[['variant', 'strategy', 'period', 'WAPE', 'relative_bias', 'WAPE + |Relative Bias|', 'selected_on_val']].to_string(index=False), flush=True)
        finalize_calibration(frozen)
    print('Независимая оценка test получена до полного переобучения. Все финальные модели обучены на всей доступной размеченной истории. Test уже изучался в проекте.', flush=True)


In [ ]:
original_shrinkage_phase = phase_forecasts

class GPUTrainingIterator(xgb.DataIter):
    def __init__(self, train, indices, segmentation):
        self.train = train
        self.indices = indices
        self.segmentation = segmentation
        self.position = 0
        self.device_batch = None
        super().__init__(release_data=True)

    def reset(self):
        self.position = 0
        self.device_batch = None

    def next(self, input_data):
        if self.position >= len(self.indices):
            return False
        rows = self.indices[self.position:self.position + GPU_BATCH_ROWS]
        self.position += len(rows)
        x = gpu_features(self.train['x'][rows], self.train['names'])
        scales = self.segmentation['scale'][self.train['route_index'][rows]]
        y = cp.asarray(self.train['y'][rows] / scales, dtype=cp.float32)
        weight = cp.asarray(scales / self.weight_scale, dtype=cp.float32)
        self.device_batch = (x, y, weight)
        input_data(data=x, label=y, weight=weight, feature_names=self.feature_names, feature_types=self.feature_types)
        return True


def gpu_features(x, names):
    values = cp.asarray(x, dtype=cp.float32)
    route = values[:, names.index('rich_route_index')].reshape(-1, 1)
    return cp.concatenate([values, route], axis=1)


def require_gpu():
    global cp
    if TASK_TYPE != 'GPU':
        raise ValueError('Обучение разрешено только на GPU')
    try:
        import cupy as cp
        if cp.cuda.runtime.getDeviceCount() < 1:
            raise RuntimeError('Нет CUDA GPU')
        x = cp.arange(16, dtype=cp.float32).reshape(8, 2)
        probe = xgb.QuantileDMatrix(x, label=cp.arange(8, dtype=cp.float32), max_bin=GPU_MAX_BIN)
        model = xgb.train(dict(device='cuda:0', tree_method='hist', max_bin=GPU_MAX_BIN, max_depth=1), probe, num_boost_round=1)
        if not json.loads(model.save_config())['learner']['generic_param']['device'].startswith('cuda'):
            raise RuntimeError('CPU fallback')
        del model, probe, x
    except Exception as error:
        raise RuntimeError('Нужны рабочие CuPy/CUDA и CUDA-сборка XGBoost с QuantileDMatrix(DataIter). Включите GPU в Kaggle. CPU fallback отключён.') from error
    print('CUDA data/model проверены:', xgb.__version__, cp.cuda.runtime.memGetInfo(), flush=True)


def fit_model(train, segmentation, family, k, settings):
    labels = segmentation['labels'][k][train['route_index']]
    specification = PARAMETERS[ACTIVE['parameter_set']][str(ACTIVE_H)]
    feature_names = list(train['names']) + ['route_code']
    feature_types = ['float'] * len(train['names']) + ['c']
    params = dict(specification['params'], objective='reg:absoluteerror', eval_metric='mae',
                  device='cuda:0', tree_method='hist', max_bin=GPU_MAX_BIN,
                  max_cat_to_onehot=1, nthread=THREADS, seed=SEED)
    models = {}
    for group in np.unique(labels):
        indices = np.arange(len(labels)) if group == -1 else np.flatnonzero(labels == group)
        iterator = GPUTrainingIterator(train, indices, segmentation)
        iterator.weight_scale = float(segmentation['scale'][train['route_index']].mean())
        iterator.feature_names, iterator.feature_types = feature_names, feature_types
        started = time.monotonic()
        try:
            matrix = xgb.QuantileDMatrix(iterator, max_bin=GPU_MAX_BIN, max_quantile_batches=GPU_QUANTILE_BATCHES,
                                        enable_categorical=True)
            model = xgb.train(params, matrix, num_boost_round=specification['trees'])
        except Exception as error:
            raise RuntimeError('Ошибка CUDA-обучения. При нехватке VRAM уменьшите GPU_BATCH_ROWS/GPU_MAX_BIN или оставьте MODEL_CLUSTERS=(8,). CPU fallback не используется.') from error
        if not json.loads(model.save_config())['learner']['generic_param']['device'].startswith('cuda'):
            raise RuntimeError('XGBoost переключился на CPU')
        models[int(group)] = dict(model=model, scaler=None)
        del matrix, iterator
        gc.collect()
        cp.get_default_memory_pool().free_all_blocks()
        print('GPU h', ACTIVE_H, 'k', k, 'group', int(group), 'rows', len(indices), 'seconds', round(time.monotonic() - started, 1), flush=True)
    return dict(models=models, k=k, residual=False, feature_names=train['names'], experiment=ACTIVE)


def predict_model(bundle, data, segmentation):
    labels = segmentation['labels'][bundle['k']][data['route_index']]
    prediction = np.empty(len(labels), dtype=np.float64)
    for group, item in bundle['models'].items():
        indices = np.flatnonzero(labels == group)
        item['model'].set_param(dict(device='cuda:0'))
        for start in range(0, len(indices), GPU_BATCH_ROWS):
            rows = indices[start:start + GPU_BATCH_ROWS]
            x = gpu_features(data['x'][rows], bundle['feature_names'])
            result = item['model'].inplace_predict(x, validate_features=False)
            if not isinstance(result, cp.ndarray):
                raise RuntimeError('Предсказание не вернуло CUDA-массив')
            prediction[rows] = cp.asnumpy(result)
            del x, result
    return np.maximum(prediction * segmentation['scale'][data['route_index']], 0)


def calibration_variants():
    return [(f'xgb_k{k}', k) for k in MODEL_CLUSTERS]


previous_shrinkage_fit = fit_factors


def fit_factors(frame, strategy):
    if not strategy.startswith('shrink_recent_'):
        return previous_shrinkage_fit(frame, strategy)
    parts = strategy.split('_')
    half_life, alpha = float(parts[2]), float(parts[3])
    long_ratio = scalar_ratio(frame)
    age = (frame.cutoff.max() - frame.cutoff).dt.total_seconds().to_numpy() / 86400
    weights = np.exp2(-age / half_life)
    recent_ratio = float(np.sum(frame.y * weights) / np.sum(frame.y_pred * weights))
    ratio = float(np.clip((1 - alpha) * long_ratio + alpha * recent_ratio, *SCALE_BOUNDS))
    return {str(h): ratio for h in HORIZONS}


CALIBRATION_STRATEGIES = ('raw', 'fixed_1.05', 'global', 'recent_global_3', 'recent_global_7', 'recent_global_14') + tuple(
    f'shrink_recent_{half_life}_{alpha}' for half_life in (3, 7, 14) for alpha in (.25, .5, .75))


def phase_forecasts(stage, boundary, query_start, query_end, frozen=None):
    if stage != 'calibration_fit':
        return original_shrinkage_phase(stage, boundary, query_start, query_end, frozen)
    boundaries = pd.date_range(query_start, query_end, freq='7D').tolist()
    if boundaries[-1] != query_end:
        boundaries.append(query_end)
    root = RUN
    block_results = []
    for index, (start, end) in enumerate(zip(boundaries[:-1], boundaries[1:])):
        globals()['RUN'] = root / f'oof_week_{index}'
        RUN.mkdir(exist_ok=True)
        try:
            block_results.append(original_shrinkage_phase(stage, start, start, end, frozen))
        finally:
            globals()['RUN'] = root
    for h in HORIZONS:
        for name, k in calibration_variants():
            frames = [pd.read_parquet(root / f'oof_week_{i}' / f'{stage}_h{h}_{name}.parquet') for i in range(len(block_results))]
            pd.concat(frames, ignore_index=True).to_parquet(root / f'{stage}_h{h}_{name}.parquet', index=False)
    return pd.concat(block_results, ignore_index=True)


In [ ]:
def show_metrics(rows, filename):
    table = pd.DataFrame(rows)
    table.to_csv(RUN / filename, index=False)
    display(table)
    print(table.to_string(index=False), flush=True)
    return table


def scope_origins(h, boundary, start, end):
    positions = np.arange(len(TIMES) - h)
    train = positions[TIMES[positions + h] < boundary]
    query = positions[(TIMES[positions] >= start) & (TIMES[positions + h] < end)
                      & (TIMES[positions].hour == 10) & (TIMES[positions].minute == 30)]
    if not len(train) or not len(query) or TIMES[train + h].max() >= boundary:
        raise ValueError('Нарушены временные границы')
    return train, query


def base_forecast(stage, boundary, start, end, variants, future=False):
    segmentation = segment_routes(boundary, stage)
    rows, futures = [], {}
    for h in HORIZONS:
        train_origins, query_origins = scope_origins(h, boundary, start, end)
        train = calibration_data(h, train_origins, segmentation)
        for name, k in variants:
            model_path = RUN / f'base_{stage}_h{h}_{name}.joblib'
            if model_path.exists():
                bundle = joblib.load(model_path)
                if bundle.get('fit_boundary') != str(boundary) or bundle.get('fit_code') != CODE_HASH:
                    raise ValueError('Несовместимый checkpoint')
            else:
                bundle = fit_model(train, segmentation, 'xgb', k, {})
                bundle.update(fit_boundary=str(boundary), fit_code=CODE_HASH)
                temporary = model_path.with_suffix('.tmp')
                joblib.dump(bundle, temporary, compress=3)
                temporary.replace(model_path)
            raw_train = predict_model(bundle, train, segmentation)
            query = calibration_data(h, query_origins, segmentation)
            raw_query = predict_model(bundle, query, segmentation)
            for period, y, pred in [('train', train['y'], raw_train), ('test' if stage == 'train_val' else stage, query['y'], raw_query)]:
                rows.append(dict(stage=stage, variant=name, h=h, period=period, method='raw', **metrics(y, pred)))
            prediction_frame(h, query, raw_query).to_parquet(RUN / f'{stage}_h{h}_{name}.parquet', index=False)
            if future:
                x, names = features(h, np.array([len(TIMES) - 1]), segmentation)
                if names != bundle['feature_names']:
                    raise ValueError('Нарушен порядок признаков')
                raw = predict_model(bundle, dict(x=x, route_index=np.arange(len(ROUTES))), segmentation)
                futures.setdefault(name, []).append(pd.DataFrame(dict(route_id=ROUTES, cutoff=TIMES[-1],
                    timestamp=TIMES[-1] + h * STEP, h=h, calibration_group=CAL_GROUP_LABELS, y_pred=raw)))
                del x, raw
            show_metrics(rows[-2:], f'fit_report_{stage}_h{h}_{name}.csv')
            del bundle, query, raw_train, raw_query
            gc.collect()
            cp.get_default_memory_pool().free_all_blocks()
        del train
        gc.collect()
    show_metrics(rows, f'base_{stage}_metrics_by_horizon.csv')
    return {name: pd.concat(parts, ignore_index=True) for name, parts in futures.items()}


def read_oof(stage, name):
    parts = [pd.read_parquet(RUN / f'{stage}_h{h}_{name}.parquet') for h in HORIZONS]
    frame = pd.concat(parts, ignore_index=True)
    if frame.duplicated(['route_id', 'cutoff', 'h']).any():
        raise ValueError('Повторные OOF-прогнозы')
    return frame


def gpu_ratio(frame, half_life=None, alpha=1.):
    y = cp.asarray(frame.y.to_numpy(), dtype=cp.float64)
    pred = cp.asarray(frame.y_pred.to_numpy(), dtype=cp.float64)
    if float(cp.sum(pred)) <= 0 or float(cp.sum(y)) <= 0:
        raise ValueError('Нет положительного объёма для калибровки')
    long_ratio = cp.sum(y) / cp.sum(pred)
    if half_life is None:
        ratio = long_ratio
    else:
        age = cp.asarray((frame.cutoff.max() - frame.cutoff).dt.total_seconds().to_numpy() / 86400)
        weight = cp.exp2(-age / half_life)
        recent_ratio = cp.sum(y * weight) / cp.sum(pred * weight)
        ratio = (1 - alpha) * long_ratio + alpha * recent_ratio
    return float(cp.clip(ratio, *SCALE_BOUNDS))


def smoothing_factor(frame, strategy):
    if strategy == 'raw':
        return 1.
    if strategy == 'fixed_1.05':
        return 1.05
    if strategy == 'global':
        return gpu_ratio(frame)
    if strategy.startswith('recent_global_'):
        return gpu_ratio(frame, float(strategy.rsplit('_', 1)[-1]))
    _, _, half_life, alpha = strategy.split('_')
    return gpu_ratio(frame, float(half_life), float(alpha))


def save_submission(frame, prediction, filename):
    output = REQUESTS.merge(frame[['route_id', 'timestamp']].assign(y_pred=prediction), on=['route_id', 'timestamp'],
                            how='left', sort=False, validate='one_to_one')
    if not output.id.equals(REQUESTS.id) or not np.isfinite(output.y_pred).all() or (output.y_pred < 0).any():
        raise ValueError('Некорректный submission')
    output[['id', 'y_pred']].to_csv(RUN / filename, index=False)


def meta_features(frame):
    origins = TIMES.get_indexer(pd.DatetimeIndex(frame.cutoff))
    route_index = pd.Index(ROUTES).get_indexer(frame.route_id)
    if (origins < 0).any() or (route_index < 0).any():
        raise ValueError('Неизвестный cutoff или маршрут метамодели')
    scale = META_SEGMENTATION['scale'][route_index]
    raw = frame.y_pred.to_numpy()
    current = VALUES[TARGET][origins, route_index]
    columns = ['meta_raw_normalized', 'meta_log_raw', 'meta_log_scale', 'meta_target_current',
               'meta_target_mean_4', 'meta_target_mean_48', 'meta_target_delta_1h',
               'meta_status_1', 'meta_status_2', 'meta_status_3', 'meta_horizon', 'meta_group']
    matrix = np.column_stack([raw / scale, np.log1p(raw), np.log1p(scale), current / scale,
        ROLL[4][origins, route_index] / scale, ROLL[48][origins, route_index] / scale,
        (current - VALUES[TARGET][np.maximum(origins - 2, 0), route_index]) / scale,
        *[VALUES[f'status_{i}'][origins, route_index] / scale for i in (1, 2, 3)],
        frame.h.to_numpy() - 1, META_LABELS[route_index] + 1]).astype(np.float32)
    if not np.isfinite(matrix).all():
        raise ValueError('Неконечные метапризнаки')
    return matrix, columns


def fit_meta(frame, config):
    x, columns = meta_features(frame)
    scales = META_SEGMENTATION['scale'][pd.Index(ROUTES).get_indexer(frame.route_id)]
    valid = frame.y_pred.to_numpy() > np.maximum(scales * META_MIN_PRED_SCALE, 1.)
    if not valid.any():
        raise ValueError('Нет положительных прогнозов для обучения коэффициента')
    raw = cp.asarray(frame.y_pred.to_numpy()[valid], dtype=cp.float32)
    target = cp.asarray(frame.y.to_numpy()[valid], dtype=cp.float32) / raw
    weights = raw / cp.mean(raw)
    device_x = cp.asarray(x[valid], dtype=cp.float32)
    types = ['float'] * len(columns)
    for c in ('meta_horizon', 'meta_group'):
        types[columns.index(c)] = 'c'
    matrix = xgb.QuantileDMatrix(device_x, label=target, weight=weights, feature_names=columns,
                                feature_types=types, enable_categorical=True, max_bin=GPU_MAX_BIN)
    settings = dict(config)
    trees = settings.pop('trees')
    model = xgb.train(dict(settings, device='cuda:0', tree_method='hist', objective='reg:squarederror',
                     max_bin=GPU_MAX_BIN, max_cat_to_onehot=1, nthread=THREADS, seed=SEED), matrix, num_boost_round=trees)
    if not json.loads(model.save_config())['learner']['generic_param']['device'].startswith('cuda'):
        raise RuntimeError('Метамодель переключилась на CPU')
    result = dict(model=model, config=config, feature_names=columns, fallback=gpu_ratio(frame))
    del matrix, device_x, raw, target, weights
    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()
    return result


def meta_coefficients(model, frame):
    x, columns = meta_features(frame)
    if columns != model['feature_names']:
        raise ValueError('Нарушен порядок метапризнаков')
    factors = np.empty(len(frame), dtype=np.float64)
    model['model'].set_param(dict(device='cuda:0'))
    for start in range(0, len(frame), GPU_BATCH_ROWS):
        batch = cp.asarray(x[start:start + GPU_BATCH_ROWS], dtype=cp.float32)
        values = model['model'].inplace_predict(batch, validate_features=False)
        if not isinstance(values, cp.ndarray):
            raise RuntimeError('Метапредсказания не на CUDA')
        factors[start:start + len(batch)] = cp.asnumpy(cp.clip(values, *SCALE_BOUNDS))
    scale = META_SEGMENTATION['scale'][pd.Index(ROUTES).get_indexer(frame.route_id)]
    small = frame.y_pred.to_numpy() <= np.maximum(scale * META_MIN_PRED_SCALE, 1.)
    factors[small] = model['fallback']
    return factors


def apply_meta(model, frame, strength):
    learned = meta_coefficients(model, frame)
    return np.clip((1 - strength) * model['fallback'] + strength * learned, *SCALE_BOUNDS)


def fit_correction(frame, choice):
    if choice['kind'] == 'scalar':
        return dict(kind='scalar', factor=smoothing_factor(frame, choice['strategy']))
    return dict(kind='meta', model=fit_meta(frame, META_CONFIGS[choice['config']]), strength=choice['strength'])


def predict_correction(correction, frame):
    factor = np.full(len(frame), correction['factor']) if correction['kind'] == 'scalar' else apply_meta(correction['model'], frame, correction['strength'])
    return frame.y_pred.to_numpy() * factor, factor


def choose_correction():
    rows = []
    for name, k in calibration_variants():
        fit, select = read_oof('calibration_fit', name), read_oof('calibration_select', name)
        if fit.timestamp.max() >= CAL_SPLIT or select.timestamp.max() >= TEST_START:
            raise ValueError('Выбор калибровки пересекает будущий период')
        for strategy in CALIBRATION_STRATEGIES:
            factor = smoothing_factor(fit, strategy)
            for period, frame in [('calibration_fit', fit), ('calibration_select', select)]:
                rows.append(dict(variant=name, k=k, kind='scalar', strategy=strategy, config=-1, strength=0.,
                                  period=period, **metrics(frame.y.to_numpy(), frame.y_pred.to_numpy() * factor)))
        if CORRECTION_MODE == 'meta':
            for index, config in enumerate(META_CONFIGS):
                model = fit_meta(fit, config)
                factors = {period: meta_coefficients(model, frame) for period, frame in [('calibration_fit', fit), ('calibration_select', select)]}
                report = []
                for strength in META_STRENGTHS:
                    for period, frame in [('calibration_fit', fit), ('calibration_select', select)]:
                        factor = np.clip((1 - strength) * model['fallback'] + strength * factors[period], *SCALE_BOUNDS)
                        row = dict(variant=name, k=k, kind='meta', strategy='learned_coefficient', config=index,
                                   strength=strength, period=period, **metrics(frame.y.to_numpy(), frame.y_pred.to_numpy() * factor))
                        rows.append(row)
                        report.append(row)
                show_metrics(report, f'meta_fit_report_{name}_{index}.csv')
                joblib.dump(model, RUN / f'meta_selection_{name}_{index}.joblib', compress=3)
                del model
                gc.collect()
    show_metrics(rows, 'calibration_selection.csv')
    selected = min([r for r in rows if r['period'] == 'calibration_select'], key=lambda r: r['score'])
    frozen = dict(selected=selected, selection_end_exclusive=str(TEST_START), test_used=False,
                  protocol='three_weekly_oof_fit_then_one_week_selection', mode=CORRECTION_MODE)
    write_json(RUN / 'calibration_frozen.json', frozen)
    print('Выбор ДО test:', selected, flush=True)
    return selected


def full_refit_future(choice):
    segmentation = segment_routes(TIMES[-1] + STEP, 'full')
    parts, rows = [], []
    for h in HORIZONS:
        train = calibration_data(h, np.arange(len(TIMES) - h), segmentation)
        model_path = RUN / f'base_full_h{h}_{choice["variant"]}.joblib'
        if model_path.exists():
            bundle = joblib.load(model_path)
            if bundle.get('fit_code') != CODE_HASH or bundle.get('fit_boundary') != str(TIMES[-1] + STEP):
                raise ValueError('Неверный full checkpoint')
        else:
            bundle = fit_model(train, segmentation, 'xgb', choice['k'], {})
            bundle.update(fit_code=CODE_HASH, fit_boundary=str(TIMES[-1] + STEP))
            temporary = model_path.with_suffix('.tmp')
            joblib.dump(bundle, temporary, compress=3)
            temporary.replace(model_path)
        raw = predict_model(bundle, train, segmentation)
        rows.append(dict(h=h, period='full_in_sample', method='raw', **metrics(train['y'], raw)))
        del train, raw
        gc.collect()
        x, names = features(h, np.array([len(TIMES) - 1]), segmentation)
        prediction = predict_model(bundle, dict(x=x, route_index=np.arange(len(ROUTES))), segmentation)
        parts.append(pd.DataFrame(dict(route_id=ROUTES, cutoff=TIMES[-1], timestamp=TIMES[-1] + h * STEP,
                                      h=h, calibration_group=CAL_GROUP_LABELS, y_pred=prediction)))
        show_metrics(rows[-1:], f'full_fit_report_h{h}.csv')
        historical = pd.read_csv(RUN / 'independent_metrics.csv')
        print('Оценка train/test до full refit:', flush=True)
        print(historical.to_string(index=False), flush=True)
        del bundle, x
        gc.collect()
        cp.get_default_memory_pool().free_all_blocks()
    show_metrics(rows, 'full_in_sample_metrics.csv')
    return pd.concat(parts, ignore_index=True)


def run_dual_submissions():
    global ACTIVE, PIPELINE_VERSION, CAT_CONFIGS, CAL_SPLIT, CAL_GROUP_LABELS, META_SEGMENTATION, META_LABELS, FEATURE_NS
    ACTIVE = EXPERIMENTS[0]
    PIPELINE_VERSION = 'dual_gpu_calibration_v1_' + CORRECTION_MODE
    CAT_CONFIGS = [dict(experiment=ACTIVE, parameters=PARAMETERS, mode=CORRECTION_MODE)]
    with threadpool_limits(limits=THREADS):
        initialize_calibration()
        META_LABELS = CAL_GROUP_LABELS.copy()
        write_json(RUN / 'protocol.json', dict(mode=CORRECTION_MODE, parameter_sources=PARAMETER_SOURCES,
            gpu_data='CuPy batches -> CUDA QuantileDMatrix', test_start=str(TEST_START), val_start=str(VAL_START),
            calibration_split=str(CAL_SPLIT), model_clusters=list(MODEL_CLUSTERS),
            meta_configs=META_CONFIGS, strengths=list(META_STRENGTHS), scale_bounds=list(SCALE_BOUNDS),
            test_used_for_selection=False, train_val_submission='no_local_test_in_fit', full_submission='all_labelled_history'))
        variants = calibration_variants()
        for index in range(3):
            start = VAL_START + pd.Timedelta(days=index * 7)
            stage = f'oof_week_{index}'
            base_forecast(stage, start, start, start + pd.Timedelta(days=7), variants)
        for h in HORIZONS:
            for name, k in variants:
                parts = [pd.read_parquet(RUN / f'oof_week_{i}_h{h}_{name}.parquet') for i in range(3)]
                pd.concat(parts, ignore_index=True).to_parquet(RUN / f'calibration_fit_h{h}_{name}.parquet', index=False)
        base_forecast('calibration_select', CAL_SPLIT, CAL_SPLIT, TEST_START, variants)
        META_SEGMENTATION = segment_routes(VAL_START, 'meta_fixed')
        meta_feature_ns = FEATURE_NS
        choice = choose_correction()
        all_val = pd.concat([read_oof(s, choice['variant']) for s in ('calibration_fit', 'calibration_select')], ignore_index=True)
        correction = fit_correction(all_val, choice)
        joblib.dump(correction, RUN / 'correction_train_val.joblib', compress=3)
        write_json(RUN / 'test_correction_frozen.json', dict(choice=choice, fit_through=str(all_val.timestamp.max()), test_used=False))
        futures = base_forecast('train_val', TEST_START, TEST_START, TEST_END, [(choice['variant'], choice['k'])], future=True)
        FEATURE_NS = meta_feature_ns
        test = read_oof('train_val', choice['variant'])
        reports = []
        for period, frame in [('meta_train_oof_in_sample', all_val), ('test', test)]:
            calibrated, factors = predict_correction(correction, frame)
            for method, prediction in [('raw', frame.y_pred.to_numpy()), ('calibrated', calibrated)]:
                reports.append(dict(period=period, method=method, **metrics(frame.y.to_numpy(), prediction)))
            frame.assign(calibrated=calibrated, factor=factors).to_parquet(RUN / f'{period}_calibration_predictions.parquet', index=False)
        show_metrics(reports, 'independent_metrics.csv')
        future = futures[choice['variant']]
        pred, factors = predict_correction(correction, future)
        save_submission(future, pred, 'submission_train_val.csv')
        save_submission(future, future.y_pred.to_numpy(), 'submission_train_val_raw.csv')
        future.assign(calibrated=pred, factor=factors).to_parquet(RUN / 'future_train_val_predictions.parquet', index=False)
        write_json(RUN / 'evaluation_complete.json', dict(choice=choice, test_evaluated_before_full_refit=True,
            train_val_model_target_end_exclusive=str(TEST_START), test_used_for_selection=False))
        all_oof = pd.concat([all_val, test], ignore_index=True)
        final_correction = fit_correction(all_oof, choice)
        joblib.dump(final_correction, RUN / 'correction_full.joblib', compress=3)
        final_oof_prediction, final_oof_factor = predict_correction(final_correction, all_oof)
        final_meta_report = dict(period='full_oof_calibrator_in_sample', method='calibrated',
                                **metrics(all_oof.y.to_numpy(), final_oof_prediction))
        show_metrics(reports + [final_meta_report], 'final_calibrator_fit_report.csv')
        all_oof.assign(calibrated=final_oof_prediction, factor=final_oof_factor).to_parquet(RUN / 'full_oof_calibrator_in_sample_predictions.parquet', index=False)
        final_future = full_refit_future(choice)
        FEATURE_NS = meta_feature_ns
        pred, factors = predict_correction(final_correction, final_future)
        save_submission(final_future, pred, 'submission_full.csv')
        save_submission(final_future, final_future.y_pred.to_numpy(), 'submission_full_raw.csv')
        final_future.assign(calibrated=pred, factor=factors).to_parquet(RUN / 'future_full_predictions.parquet', index=False)
        note = 'Test оценён до полного refit, выбор модели/калибровки только по val. Этот test уже изучался в проекте. submission_train_val использует базы target timestamp < TEST_START и калибратор только на OOF val. submission_full использует базы со всей истории и калибратор на всех доступных OOF val+бывший test без повторного выбора. Мета-признаки используют фиксированные статистики до val для одинакового масштаба метамодели. Все fit и model inference получают CUDA-матрицы; parquet, признаки и отчёты готовятся на CPU.'
        (RUN / 'evaluation_note.txt').write_text(note)
        show_metrics(reports, 'final_report_historical_metrics.csv')
        print(note, flush=True)
    archive_path = RUN.with_suffix('.zip')
    with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(RUN.rglob('*')):
            if path.is_file():
                archive.write(path, path.relative_to(RUN.parent))
    print('Основные сабмиты:', RUN / 'submission_train_val.csv', RUN / 'submission_full.csv', flush=True)
    print('Итоговый архив:', archive_path, flush=True)


In [ ]:
def past(array, positions):
    positions = np.asarray(positions, dtype=np.int64)
    valid = (positions >= 0) & (positions < len(array))
    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)
    result[~valid] = np.nan
    return result

def average_available(matrices):
    total = np.zeros_like(matrices[0], dtype=np.float32)
    count = np.zeros_like(total)
    for matrix in matrices:
        valid = np.isfinite(matrix)
        total += np.where(valid, matrix, 0)
        count += valid
    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)

def rolling(array, steps):
    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)

def common_features(origins):
    common = {}
    for k in (0, 1, 2, 4):
        common[f'target_back_{k}'] = past(target, origins - k)
    for name, array in values.items():
        for steps in (4, 12):
            common[f'{name}_mean_{steps}'] = means[name, steps][origins]
        if name in STATUSES:
            common[f'{name}_latest'] = array[origins]
            if name in STATUSES[:3]:
                common[f'{name}_change_1h'] = array[origins] - past(array, origins - 2)
    for name, array in extra_signals.items():
        common[name] = array[origins]
    for lag in (48, 96):
        common[f'target_1h_lag_{lag}'] = past(target, origins - lag)
    common['warehouse_total_load_lag_48'] = past(extra_signals['warehouse_total_load'], origins - 48)
    common['status_6_diff_hourly_lag_48'] = past(hub_signal, origins - 48)
    common['target_vs_mean_6h'] = target[origins] - means[TARGET, 12][origins]
    recent = means[TARGET, 4][origins]
    usual = average_available([past(means[TARGET, 4], origins - day * 48) for day in range(1, 8)])
    common['morning_level_ratio'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)
    return common
CORRELATION_COLUMNS = ['selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available']

def calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):
    grid = pd.DatetimeIndex(timestamps)
    route_ids = np.asarray(route_ids)
    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))
    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):
        raise ValueError('Ожидаются два массива [время, маршрут] одной формы')
    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):
        raise ValueError('Для выбора лага нужна упорядоченная получасовая сетка без дубликатов')
    if len(np.unique(route_ids)) != len(route_ids):
        raise ValueError('Идентификаторы маршрутов повторяются')
    steps = float(max_lookback_hours) * 2
    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):
        raise ValueError('Граница поиска должна задавать целое число получасов, не меньше двух')
    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:
        raise ValueError('min_pairs должен быть целым числом >=3')
    if not 0 <= min_correlation <= 1 or route_block_size < 1:
        raise ValueError('Некорректные параметры выбора корреляции')
    max_lag = int(steps)
    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side='left'))
    anchors = np.arange(max_lag + 2, stop)
    best_corr = np.full(len(route_ids), np.nan)
    best_lag = np.full(len(route_ids), np.nan)
    best_pairs = np.zeros(len(route_ids), dtype=np.int32)
    for start in range(0, len(route_ids), route_block_size):
        finish = min(start + route_block_size, len(route_ids))
        x = target_values[anchors, start:finish].astype(np.float64)
        for lag in range(2, max_lag + 1):
            z = status6_diff[anchors - lag, start:finish].astype(np.float64)
            valid = np.isfinite(x) & np.isfinite(z)
            count = valid.sum(axis=0)
            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))
            divisor = np.maximum(count, 1)
            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))
            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor
            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)
            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)
            denominator = np.sqrt(vx * vz)
            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))
            corr = np.clip(corr, -1, 1)
            previous = best_corr[start:finish]
            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))
            best_corr[start:finish][better] = corr[better]
            best_lag[start:finish][better] = lag
            best_pairs[start:finish][better] = count[better]
    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)
    return pd.DataFrame({'route_id': route_ids, 'selected_status6_lag_steps': np.where(accepted, best_lag, np.nan), 'selected_status6_correlation': best_corr, 'selected_status6_pair_count': best_pairs, 'status6_lag_valid': accepted.astype(np.int8)})

def add_correlation_features(frame, h, lag_table, chunk_rows=200000):
    if h not in range(1, 9) or chunk_rows < 1:
        raise ValueError('Некорректный горизонт или размер блока')
    by_route = lag_table.set_index('route_id').reindex(routes)
    route_index = pd.Index(routes)
    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)
    quality = {name: by_route[name].to_numpy(dtype=np.float32) for name in CORRELATION_COLUMNS[:4]}
    for name in CORRELATION_COLUMNS:
        frame[name] = np.full(len(frame), np.nan, dtype=np.float32)
    column_positions = {name: frame.columns.get_loc(name) for name in CORRELATION_COLUMNS}
    for start in range(0, len(frame), chunk_rows):
        finish = min(start + chunk_rows, len(frame))
        block = frame.iloc[start:finish]
        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))
        route = route_index.get_indexer(block.route_id)
        if (origin < 0).any() or (route < 0).any():
            raise ValueError('Строка признаков не сопоставлена с историей')
        selected = lag[route]
        source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64)
        available = np.isfinite(selected) & (selected >= h) & (source >= 0) & (source <= origin) & (source < len(times))
        signal = hub_signal[np.clip(source, 0, len(times) - 1), route].astype(np.float32)
        available &= np.isfinite(signal)
        for name, array in quality.items():
            frame.iloc[start:finish, column_positions[name]] = array[route]
        frame.iloc[start:finish, column_positions['lagged_status6_signal']] = np.where(available, signal, np.nan)
        frame.iloc[start:finish, column_positions['lagged_status6_signal_available']] = available.astype(np.float32)
    return frame

def horizon_dataset(h, origins):
    slot = origins + h
    dates = times[origins] + h * STEP
    frame = pd.DataFrame({'cutoff': times[origins].repeat(len(routes)), 'timestamp': dates.repeat(len(routes)), 'route_id': np.tile(routes, len(origins))})
    for name, matrix in common_features(origins).items():
        frame[name] = matrix.ravel()
    calendar = {'hour': dates.hour, 'minute': dates.minute, 'hour_float': dates.hour + dates.minute / 60.0, 'dayofweek': dates.dayofweek, 'is_weekend': (dates.dayofweek >= 5).astype(np.int8)}
    calendar['hour_sin'] = np.sin(2 * np.pi * calendar['hour_float'] / 24)
    calendar['hour_cos'] = np.cos(2 * np.pi * calendar['hour_float'] / 24)
    calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
    calendar['weekday_cos'] = np.cos(2 * np.pi * dates.dayofweek / 7)
    for name, array in calendar.items():
        frame[name] = np.repeat(array, len(routes))
    frame = add_correlation_features(frame, h, validation_lag_table)
    for days in (1, 2, 7):
        frame[f'slot_days_back_{days}'] = past(target, slot - days * 48).ravel()
    for name, days in [('slot_daily7', range(1, 8)), ('slot_weekly4', (7, 14, 21, 28))]:
        frame[name] = average_available([past(target, slot - d * 48) for d in days]).ravel()
    profile = np.zeros((len(origins), len(routes)))
    counts = np.zeros_like(profile)
    for day in range(1, 29):
        level = past(level24, origins - day * 48)
        adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)
        profile += np.nan_to_num(adjusted)
        counts += np.isfinite(adjusted)
    frame['profile28'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()
    if h == 1:
        lower = np.zeros_like(target[origins])
        upper = past(target, origins - 2)
        for position in (origins - 1, origins):
            total = past(target, position)
            lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))
        seasonal = frame[['slot_daily7', 'slot_weekly4']].mean(axis=1).to_numpy()
        frame['overlap'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)
    else:
        frame['overlap'] = np.nan
    numeric = frame.columns.difference(['cutoff', 'timestamp', 'route_id'])
    return frame.astype({name: 'float32' for name in numeric})

def add_pilot_features(frame, h, origins):
    current = target[origins]
    level = level24[origins]
    frame['level24_feature'] = level.ravel()
    frame['target_level24_logratio'] = (np.log1p(current) - np.log1p(level)).ravel()
    frame['mean2h_level24_logratio'] = (np.log1p(means[TARGET, 4][origins]) - np.log1p(level)).ravel()
    frame['profile28_minus_daily7'] = (frame.profile28 - frame.slot_daily7).astype(np.float32)
    return frame.astype({c: 'float32' for c in [*SEASONAL_FEATURES, *STATUS_FEATURES]})
SEASONAL_FEATURES = ['profile28', 'level24_feature', 'target_level24_logratio', 'mean2h_level24_logratio', 'profile28_minus_daily7']
STATUS_FEATURES = []

def extend_robust(data):
    new = {}
    floor = 1 / np.maximum(data.scale.to_numpy(dtype=float), 1)
    for j in range(1, 7):
        name = f'status_{j}'
        level = np.maximum(data[f'long_{name}_mean48'].to_numpy(dtype=float), floor)
        week = np.maximum(data[f'long_{name}_mean336'].to_numpy(dtype=float), floor)
        for suffix, key in [('current', f'{name}_lag0'), ('mean4', f'{name}_mean4'), ('mean12', f'{name}_mean12'), ('delta4', f'{name}_delta4')]:
            new[f'robust_s{j}_{suffix}'] = np.clip(data[key].to_numpy() / level, -100, 100).astype(np.float32)
        new[f'robust_s{j}_level_change'] = np.clip(level / week, 0, 100).astype(np.float32)
        raw = data[f'{name}_lag0'].to_numpy(dtype=float) * data.scale.to_numpy()
        denominator = pd.Series(raw).groupby([data.cutoff, data.horizon], sort=False).transform('sum').to_numpy()
        new[f'robust_s{j}_route_share'] = np.divide(raw, np.maximum(denominator, 1)).astype(np.float32)
    for window in [4, 12]:
        new[f'robust_target_vs{window}'] = np.clip(data.target_lag_0.to_numpy() / np.maximum(data[f'target_mean_{window}'].to_numpy(), floor), 0, 100).astype(np.float32)
    return pd.concat([data, pd.DataFrame(new, index=data.index)], axis=1)

def session_shape_features(origins, h, profile_history, daily):
    T = len(times)
    R = len(routes)

    def past(a, index):
        index = np.asarray(index)
        out = a[np.clip(index, 0, T - 1)].astype(np.float32, copy=True)
        out[(index < 0) | (index >= T)] = np.nan
        return out
    scale = np.maximum(daily[origins], 1)
    common = {}
    for lag in (*range(9), 12, 24, 48, 96, 336):
        common[f'target_lag_{lag}'] = past(target, origins - lag) / scale
    for window in (4, 12, 48):
        common[f'target_mean_{window}'] = (daily if window == 48 else means['target_1h', window])[origins] / scale
    for window in (4, 12):
        common[f'target_std_{window}'] = deviations[window][origins] / scale
    for name in names[1:]:
        a = values[name]
        for lag in (0, 1, 2):
            common[f'{name}_lag{lag}'] = past(a, origins - lag) / scale
        for window in (4, 12):
            common[f'{name}_mean{window}'] = means[name, window][origins] / scale
        for lag in (2, 4):
            common[f'{name}_delta{lag}'] = (a[origins] - past(a, origins - lag)) / scale
    common['log_current_level'] = np.log1p(scale)
    for name, a in values.items():
        total = a.sum(axis=1, dtype=np.float64)
        netlevel = pd.Series(total).rolling(48, min_periods=1).mean().to_numpy()
        net12 = pd.Series(total).rolling(12, min_periods=1).mean().to_numpy()
        den = np.maximum(netlevel[origins], 1)
        for suffix, x in [('level', total[origins] / den), ('mean12', net12[origins] / den), ('delta2', (total[origins] - total[origins - 2]) / den)]:
            common[f'network_{name}_{suffix}'] = np.broadcast_to(x[:, None], scale.shape).astype(np.float32)
    common['route_network_share'] = target[origins] / np.maximum(target[origins].sum(axis=1, keepdims=True), 1)
    slot = origins + h
    dates = times[origins] + pd.Timedelta(minutes=30 * h)
    arrays = {name: a.ravel() for name, a in common.items()}
    for day in (1, 2, 3, 7, 14, 21, 28):
        a = past(target, slot - day * 48)
        arrays[f'slot_day{day}'] = (a / scale).ravel()
    for day in (1, 7):
        for offset in (-2, -1, 1, 2):
            arrays[f'slot_day{day}_offset{offset}'] = (past(target, slot - day * 48 + offset) / scale).ravel()
    stack = np.stack([past(target, slot - day * 48) for day in range(1, 8)])
    arrays['slot_daily7'] = (np.nanmean(stack, axis=0) / scale).ravel()
    stack = np.stack([past(target, slot - day * 48) for day in (7, 14, 21, 28)])
    with np.errstate(invalid='ignore'):
        weekly = np.nanmean(stack, axis=0)
    arrays['slot_weekly4'] = (weekly / scale).ravel()
    profile_sum = np.zeros_like(scale, dtype=np.float64)
    count = np.zeros_like(scale)
    for day in range(1, 29):
        a = past(profile_history, slot - day * 48) * scale / np.maximum(past(daily, origins - day * 48), 1)
        valid = np.isfinite(a)
        profile_sum += np.nan_to_num(a)
        count += valid
    profile = np.divide(profile_sum, count, out=np.zeros_like(profile_sum), where=count > 0).astype(np.float32)
    arrays['profile_norm'] = (profile / scale).ravel()
    arrays['horizon'] = np.full(len(origins) * R, h, dtype=np.int8)
    arrays['dayofweek'] = np.repeat(dates.dayofweek, R).astype(np.int8)
    arrays['origin_minute'] = np.repeat(times[origins].hour * 60 + times[origins].minute, R).astype(np.int16)
    arrays['future_hour'] = np.repeat(dates.hour + dates.minute / 60, R).astype(np.float32)
    frame = pd.DataFrame(arrays)
    frame['cutoff'] = times[origins].repeat(R)
    frame['timestamp'] = dates.repeat(R)
    frame['route_id'] = np.tile(routes, len(origins)).astype(np.int16)
    frame['scale'] = scale.ravel()
    frame['profile28'] = profile.ravel()
    frame['y'] = past(target, slot).ravel()
    extra = {}
    for name in names[1:]:
        for lag in (4, 8, 12, 24, 48, 96):
            extra[f'long_{name}_lag{lag}'] = (past(values[name], origins - lag) / scale).ravel()
        for window, series in [(48, means[name, 48]), (336, long_week[name])]:
            extra[f'long_{name}_mean{window}'] = (series[origins] / scale).ravel()
    return extend_robust(pd.concat([frame, pd.DataFrame(extra)], axis=1))

def prepare_history(branch):
    global values, target, means, level24, extra_signals, hub_signal, validation_lag_table, long_week, deviations
    values = branch
    target = values[TARGET]
    means = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}
    level24 = rolling(target, 48)
    positions = np.arange(len(times))
    extra_signals = {}
    for name in STATUSES[:3]:
        extra_signals[name + '_hourly'] = values[name] + past(values[name], positions - 1)
    extra_signals['warehouse_total_load'] = sum((extra_signals[name + '_hourly'] for name in STATUSES[:3]))
    extra_signals['warehouse_processing_rate'] = extra_signals['status_3_hourly'] / (extra_signals['status_1_hourly'] + np.float32(1e-05))
    for name in STATUSES[3:]:
        extra_signals[name + '_diff_hourly'] = values[name] - past(values[name], positions - 2)
    extra_signals['hub_buffer'] = values['status_5'] - values['status_4']
    extra_signals['hub_gate_bottleneck'] = extra_signals['status_4_diff_hourly'] - extra_signals['status_6_diff_hourly']
    extra_signals['target_acceleration'] = past(target, positions - 1) - past(target, positions - 2)
    hub_signal = extra_signals['status_6_diff_hourly']
    means.update({(name, 48): rolling(a, 48) for name, a in values.items()})
    long_week = {name: rolling(values[name], 336) for name in STATUSES}
    deviations = {w: pd.DataFrame(target).rolling(w, min_periods=2).std().to_numpy(dtype=np.float32) for w in (4, 12)}
    validation_lag_table = calculate_dynamic_shoulder(target, hub_signal, times, routes, fit_before=start_tune)

cat_columns = ['route_id', 'target_back_0', 'target_back_1', 'target_back_2', 'target_back_4', 'target_1h_mean_4', 'target_1h_mean_12', 'status_1_mean_4', 'status_1_mean_12', 'status_1_latest', 'status_1_change_1h', 'status_2_mean_4', 'status_2_mean_12', 'status_2_latest', 'status_2_change_1h', 'status_3_mean_4', 'status_3_mean_12', 'status_3_latest', 'status_3_change_1h', 'status_4_mean_4', 'status_4_mean_12', 'status_4_latest', 'status_5_mean_4', 'status_5_mean_12', 'status_5_latest', 'status_6_mean_4', 'status_6_mean_12', 'status_6_latest', 'status_1_hourly', 'status_2_hourly', 'status_3_hourly', 'warehouse_total_load', 'warehouse_processing_rate', 'status_4_diff_hourly', 'status_5_diff_hourly', 'status_6_diff_hourly', 'hub_buffer', 'hub_gate_bottleneck', 'target_acceleration', 'target_1h_lag_48', 'target_1h_lag_96', 'warehouse_total_load_lag_48', 'status_6_diff_hourly_lag_48', 'target_vs_mean_6h', 'morning_level_ratio', 'hour', 'minute', 'hour_float', 'dayofweek', 'is_weekend', 'hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos', 'selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available', 'slot_days_back_1', 'slot_days_back_2', 'slot_days_back_7', 'slot_daily7', 'slot_weekly4']
cat_trees = [1500, 1500, 1500, 1500, 1499, 1500, 1497, 1500]
anchor_columns = ['target_back_0', 'target_back_1', 'target_back_2', 'target_back_4', 'target_1h_mean_4', 'target_1h_mean_12', 'level24_feature', 'target_1h_lag_48', 'target_1h_lag_96', 'target_vs_mean_6h', 'morning_level_ratio', 'warehouse_processing_rate', 'target_acceleration']
anchor_columns += [f'status_{j}_{suffix}' for j in range(1, 7) for suffix in ('latest', 'mean_4', 'mean_12')]

def make_cat_frame(origins, h):
    return add_pilot_features(horizon_dataset(h, origins), h, origins)

def metric(actual, predicted):
    y = np.asarray(actual, dtype=float)
    p = np.asarray(predicted, dtype=float)
    if y.shape != p.shape or not np.isfinite(y).all() or (not np.isfinite(p).all()) or (y.sum() <= 0):
        raise ValueError('Неверные данные для метрики')
    error = p - y
    wape = np.abs(error).sum() / y.sum()
    bias = error.sum() / y.sum()
    return dict(rows=len(y), WAPE=wape, relative_bias=bias, score=wape + abs(bias))
log = []

def report(branch, stage, h, split, y, p):
    log.append(dict(branch=branch, stage=stage, h=h, split=split, **metric(y, p)))

def show_fit():
    display(pd.DataFrame(log[-2:]))
    pd.DataFrame(log).to_csv(out / 'metrics_after_fit.csv', index=False)

def base_predictions(branch, h):
    eligible = positions[times + h * STEP < start_tune]
    cat_origins = eligible[len(eligible) // 4:]
    columns = cat_columns + (SEASONAL_FEATURES if h in (5, 6, 8) else [])
    chunks = []
    for batch in range(0, len(cat_origins), 32):
        chunks.append(make_cat_frame(cat_origins[batch:batch + 32], h)[columns])
    train = pd.concat(chunks, ignore_index=True)
    del chunks
    cat = CatBoostRegressor(iterations=cat_trees[h - 1], depth=7, learning_rate=0.05, loss_function='MAE', task_type='GPU', devices='0', cat_features=['route_id'], random_seed=42, allow_writing_files=False, verbose=250, thread_count=4)
    cat.fit(train, target[cat_origins + h].ravel() / 100000)
    model_dir = out / 'models' / branch
    cat.save_model(str(model_dir / f'catboost_h{h}.cbm'))
    fit_prediction = np.maximum(cat.predict(train) * 100000, 0)
    report(branch, 'CatBoost', h, 'train', original_target[cat_origins + h].ravel(), fit_prediction)
    del train, fit_prediction
    f = make_cat_frame(eval_origins, h)
    frame = f[['cutoff', 'route_id', 'timestamp', 'profile28', *anchor_columns]].copy()
    frame['horizon'] = h
    frame['y'] = original_target[eval_origins + h].ravel()
    frame['fit_y'] = target[eval_origins + h].ravel()
    frame['model_prediction'] = np.maximum(cat.predict(f[columns]) * 100000, 0)
    test = frame.cutoff >= VAL_FIRST
    report(branch, 'CatBoost', h, 'val', frame.loc[test, 'y'], frame.loc[test, 'model_prediction'])
    show_fit()
    frame['raw_blend'] = frame.model_prediction if h == 1 else 0.7 * frame.model_prediction + 0.3 * frame.profile28
    del cat, f
    origins = eligible[(eligible >= 336) & np.isin(minutes[eligible], [570, 600, 630, 660, 690])]
    train = session_shape_features(origins, h, target, level24)
    cfg = meta[str(h)]
    cols = cfg['columns']
    model = lgb.LGBMRegressor(**dict(cfg['params'], n_estimators=cfg['trees'], n_jobs=4))
    scale = train.scale.to_numpy(dtype=float)
    model.fit(train[cols], target[origins + h].ravel() / scale, sample_weight=scale / scale.mean())
    model.booster_.save_model(str(model_dir / f'lightgbm_h{h}.txt'))
    fit_prediction = np.maximum(model.predict(train[cols]) * scale, 0)
    report(branch, 'LightGBM', h, 'train', original_target[origins + h].ravel(), fit_prediction)
    del train, fit_prediction
    evaluation = session_shape_features(eval_origins, h, target, level24)
    if not frame[keys].reset_index(drop=True).equals(evaluation[keys].astype(frame[keys].dtypes).reset_index(drop=True)):
        raise ValueError('Расходятся ключи между базовыми моделями')
    frame['shape'] = np.maximum(model.predict(evaluation[cols]) * evaluation.scale.to_numpy(dtype=float), 0)
    report(branch, 'LightGBM', h, 'val', frame.loc[test, 'y'], frame.loc[test, 'shape'])
    show_fit()
    return frame

def residual_features(frame):
    cols = ['route_id', 'horizon', 'raw_blend', 'model_prediction', 'profile28', *anchor_columns]
    X = frame[cols].copy()
    X['dayofweek'] = frame.cutoff.dt.dayofweek
    X['route_id'] = X.route_id.astype(str)
    X['horizon'] = X.horizon.astype(str)
    for name in ['model_prediction', 'profile28', 'target_back_0', 'target_1h_mean_4', 'target_1h_mean_12', 'level24_feature', 'target_1h_lag_48', 'target_1h_lag_96']:
        X[f'{name}_minus_base'] = X[name] - X.raw_blend
    return X

def causal_offsets(frame, initial, unshifted, fit_mask):
    errors = frame.fit_y.to_numpy() - unshifted
    offsets = np.full(len(frame), initial)
    for cutoff in frame.loc[~fit_mask, 'cutoff'].drop_duplicates():
        known = frame.timestamp < cutoff - pd.Timedelta(hours=4)
        if frame.loc[known, 'cutoff'].nunique() >= 3:
            ages = (cutoff - frame.loc[known, 'cutoff']).dt.total_seconds().to_numpy() / 86400
            offsets[frame.cutoff.eq(cutoff)] = np.average(errors[known], weights=np.exp2(-ages / 7))
    return offsets

def finish_pipeline(frame, branch, fit_before):
    fit = frame.cutoff < fit_before
    test = frame.cutoff >= fit_before
    X = residual_features(frame)
    corrected = []
    for depth, trees, l2 in [(4, 450, 30), (5, 900, 50)]:
        model = CatBoostRegressor(iterations=trees, depth=depth, learning_rate=0.03, l2_leaf_reg=l2, loss_function='MAE', cat_features=['route_id', 'horizon'], random_seed=42, thread_count=4, task_type='GPU', devices='0', allow_writing_files=False, verbose=False)
        model.fit(X.loc[fit], (frame.loc[fit, 'fit_y'] - frame.loc[fit, 'raw_blend']) / 100000)
        unshifted = frame.raw_blend.to_numpy() + model.predict(X) * 100000
        initial = np.mean(frame.loc[fit, 'fit_y'] - unshifted[fit])
        folder = out / 'models' / branch / f'corrections_before_{fit_before:%Y%m%d}'
        folder.mkdir(parents=True, exist_ok=True)
        model.save_model(str(folder / f'depth{depth}.cbm'))
        state = frame[keys + ['timestamp']].copy()
        state['residual'] = frame.fit_y.to_numpy() - unshifted
        state['initial_offset'] = float(initial)
        state['used_for_fit'] = fit.to_numpy()
        state.to_parquet(folder / f'depth{depth}_history.parquet', index=False)
        pred = np.maximum(unshifted + causal_offsets(frame, initial, unshifted, fit), 0)
        stage = f'residual_depth{depth}_fit_before_{fit_before:%m%d}'
        report(branch, stage, 'all', 'train', frame.loc[fit, 'y'], pred[fit])
        report(branch, stage, 'all', 'val', frame.loc[test, 'y'], pred[test])
        show_fit()
        corrected.append(pred)
    p = frame[keys + ['y', 'shape']].copy()
    p['volume_model'] = (corrected[0] + corrected[1]) / 2
    totals = p.groupby(['cutoff', 'horizon'])[['volume_model', 'shape']].transform('sum')
    if totals['shape'].le(0).any():
        raise ValueError('Нулевая сумма LightGBM')
    p['prediction'] = (1 - weight) * p.volume_model + weight * p['shape'] * totals.volume_model / totals['shape']
    p['branch'] = branch
    return p

In [ ]:
def read_xgb_archive(stage, boundary, end):
    with zipfile.ZipFile(XGB_ARCHIVE) as archive:
        prefix = XGB_ARCHIVE.stem + '/'
        manifest = json.loads(archive.read(prefix + 'manifest.json'))
        current = json.loads((RUN / 'manifest.json').read_text())
        previous = manifest['cat_configs'][0]
        if manifest['train_hash'] != current['train_hash'] or manifest['test_hash'] != current['test_hash'] or previous['parameters'] != PARAMETERS:
            raise ValueError('Архив XGBoost использует другие входы или параметры')
        protocol = json.loads(archive.read(prefix + 'protocol.json'))
        evaluation = json.loads(archive.read(prefix + 'evaluation_complete.json'))
        allowed = {'oof_week_2': pd.Timestamp(protocol['val_start']) + pd.Timedelta(days=14),
                   'calibration_select': pd.Timestamp(protocol['calibration_split']),
                   'train_val': pd.Timestamp(evaluation['train_val_model_target_end_exclusive'])}
        if allowed[stage] != boundary or protocol['model_clusters'] != [8] or manifest['clusters'] != list(CLUSTERS) or manifest['seed'] != SEED:
            raise ValueError('Границы или настройки кластеров архива отличаются')
        parts = [pd.read_parquet(BytesIO(archive.read(prefix + f'{stage}_h{h}_xgb_k8.parquet'))) for h in HORIZONS]
    frame = pd.concat(parts, ignore_index=True).rename(columns={'h': 'horizon', 'y_pred': 'xgb_all'})
    check_panel(frame, boundary, end)
    return frame


def check_panel(frame, start, end):
    expected = pd.MultiIndex.from_product([
        pd.date_range(start + pd.Timedelta(hours=10, minutes=30), end, freq='D', inclusive='left'),
        HORIZONS, ROUTES], names=keys)
    actual = pd.MultiIndex.from_frame(frame[keys])
    if actual.has_duplicates or len(actual) != len(expected) or len(expected.difference(actual)):
        raise ValueError('Неполная панель: ' + str(start))
    if not frame.timestamp.eq(frame.cutoff + pd.to_timedelta(frame.horizon * 30, unit='m')).all():
        raise ValueError('Некорректные timestamps целей')


def fit_five_starts(stage, boundary, end):
    segmentation = segment_routes(boundary, stage)
    reports, parts = [], []
    for h in HORIZONS:
        train_origins, query_origins = scope_origins(h, boundary, boundary, end)
        train_origins = train_origins[np.isin(minutes[train_origins], MORNING_STARTS)]
        if not len(train_origins):
            raise ValueError('Нет обучающих утренних стартов')
        train = calibration_data(h, train_origins, segmentation)
        query = calibration_data(h, query_origins, segmentation)
        path = RUN / f'{stage}_five_h{h}.joblib'
        if path.exists():
            bundle = joblib.load(path)
            if bundle['fit_boundary'] != str(boundary) or bundle['fit_code'] != CODE_HASH:
                raise ValueError('Несовместимый checkpoint')
        else:
            bundle = fit_model(train, segmentation, 'xgb', 8, {})
            bundle.update(fit_boundary=str(boundary), fit_code=CODE_HASH)
            temporary_path = path.with_suffix('.tmp')
            joblib.dump(bundle, temporary_path, compress=3)
            temporary_path.replace(path)
        train_prediction = predict_model(bundle, train, segmentation)
        query_prediction = predict_model(bundle, query, segmentation)
        parts.append(prediction_frame(h, query, query_prediction).rename(columns={'h': 'horizon', 'y_pred': 'xgb_five'}))
        rows = [dict(horizon=h, period=period, **metrics(y, pred)) for period, y, pred in
                [('train', train['y'], train_prediction), ('test' if stage == 'test' else 'val', query['y'], query_prediction)]]
        reports.extend(rows)
        display(pd.DataFrame(rows))
        pd.DataFrame(reports).to_csv(RUN / f'{stage}_five_fit_metrics.csv', index=False)
        del train, query, bundle, train_prediction, query_prediction
        gc.collect()
        cp.get_default_memory_pool().free_all_blocks()
    frame = pd.concat(parts, ignore_index=True)
    check_panel(frame, boundary, end)
    return frame


def merge_predictions(left, right, column):
    merged = left.merge(right[keys + ['timestamp', 'y', column]], on=keys, how='outer',
                        validate='one_to_one', suffixes=('', '_incoming'))
    if len(merged) != len(left) or len(merged) != len(right) or not np.array_equal(merged.y, merged.y_incoming) or not merged.timestamp.eq(merged.timestamp_incoming).all():
        raise ValueError('Не совпали ключи, факты или timestamp: ' + column)
    if not np.isfinite(merged[column]).all() or merged[column].lt(0).any():
        raise ValueError('Некорректный прогноз: ' + column)
    return merged.drop(columns=['y_incoming', 'timestamp_incoming'])


def fixed_volume_mix(frame):
    result = frame.copy()
    anchor = result.volume_model.to_numpy(dtype=float)
    group = ['cutoff', 'horizon']
    volume = result.groupby(group).volume_model.transform('sum').to_numpy(dtype=float)
    for name in ('all', 'five'):
        raw = result[f'xgb_{name}'].to_numpy(dtype=float)
        total = result.groupby(group)[f'xgb_{name}'].transform('sum').to_numpy(dtype=float)
        normalized = np.divide(raw * volume, total, out=anchor.copy(), where=total > 0)
        replacement = (1 - weight) * anchor + weight * normalized
        result[f'mix_{name}'] = .75 * result.session_prediction + .25 * replacement
    totals = result.groupby(group)[['volume_model', 'session_prediction', 'mix_all', 'mix_five']].sum()
    if not np.allclose(totals[['session_prediction', 'mix_all', 'mix_five']], totals.volume_model.to_numpy()[:, None], atol=1e-5, rtol=1e-10):
        raise ValueError('Изменился объём горизонта')
    return result


def volume_factor(frame, column, alpha):
    total_pred = float(frame[column].sum())
    if total_pred <= 0:
        raise ValueError('Нет объёма для калибровки')
    ratio = float(np.clip(frame.y.sum() / total_pred, .95, 1.05))
    return 1 + alpha * (ratio - 1)


def rows_for(frame, period, columns):
    return [dict(period=period, variant=column, **metrics(frame.y.to_numpy(), frame[column].to_numpy())) for column in columns]


def choose_on_val(val):
    shape_rows = []
    weeks = [('val_week1', val[val.cutoff < VAL_SECOND]), ('val_week2', val[val.cutoff >= VAL_SECOND])]
    columns = ('session_prediction', 'xgb_all', 'xgb_five', 'mix_all', 'mix_five')
    for period, part in [*weeks, ('val_all', val)]:
        shape_rows.extend(rows_for(part, period, columns))
    table = show_metrics(shape_rows, 'val_shape_metrics.csv')
    candidates = table[table.variant.isin(['mix_all', 'mix_five']) & table.period.isin(['val_week1', 'val_week2'])]
    selected = candidates.groupby('variant').WAPE.mean().idxmin()
    first, second = (part for _, part in weeks)
    calibration = []
    for alpha in (0., .25, .5, 1.):
        factor = volume_factor(first, selected, alpha)
        calibration.append(dict(alpha=alpha, factor_fit_on_week1=factor,
                                **metrics(second.y.to_numpy(), second[selected].to_numpy() * factor)))
    show_metrics(calibration, 'val_volume_calibration.csv')
    alpha = min(calibration, key=lambda row: row['score'])['alpha']
    selection = dict(shape=selected, alpha=alpha, factor=volume_factor(val, selected, alpha),
                     fit_through=str(val.timestamp.max()), test_used_for_selection=False)
    write_json(RUN / 'selection_frozen.json', selection)
    return selection


def make_val_anchor():
    folder = RUN / 'anchor_oof.parquet'
    if folder.exists():
        return pd.read_parquet(folder)
    prepare_history(VALUES)
    parts = [base_predictions('session6_oof', h) for h in HORIZONS]
    base = pd.concat(parts, ignore_index=True).sort_values(keys).reset_index(drop=True)
    corrected = []
    for start, end in ((VAL_FIRST, VAL_SECOND), (VAL_SECOND, TEST_START)):
        block = base[base.cutoff < end].reset_index(drop=True)
        pred = finish_pipeline(block, 'session6_oof', start)
        query = pred[pred.cutoff >= start].rename(columns={'prediction': 'session_prediction'})
        query = query.merge(block[keys + ['timestamp']], on=keys, validate='one_to_one')
        check_panel(query, start, end)
        corrected.append(query)
    val = pd.concat(corrected, ignore_index=True)
    val.to_parquet(folder, index=False)
    return val


def run_starts_comparison():
    global RUN, out, CAL_GROUP_LABELS, times, routes, positions, minutes, eval_origins
    global start_tune, original_target, target, values, names, meta, weight
    load_inputs()
    if TEST_START != pd.Timestamp('2025-10-18') or TEST_END != pd.Timestamp('2025-11-01'):
        raise ValueError('Архивы относятся к локальному test 18–31 октября')
    out = RUN
    times, routes, values = TIMES, ROUTES, VALUES
    original_target = target = VALUES[TARGET]
    names = [TARGET, *STATUSES]
    positions = np.arange(len(times))
    minutes = times.hour * 60 + times.minute
    eval_origins = positions[(times >= ANCHOR_SEED) & (times < TEST_START) & (minutes == 630)]
    start_tune = ANCHOR_SEED
    CAL_GROUP_LABELS = np.zeros(len(routes), dtype=int)
    with zipfile.ZipFile(SESSION_ARCHIVE) as archive:
        meta = json.loads(archive.read('comparison_meta.json'))
    weight = float(meta['weight'])
    (RUN / 'models' / 'session6_oof').mkdir(parents=True, exist_ok=True)
    with threadpool_limits(limits=THREADS):
        val_anchor = make_val_anchor()
        val_parts = []
        for stage, boundary, end in [('oof_week_2', VAL_FIRST, VAL_SECOND), ('calibration_select', VAL_SECOND, TEST_START)]:
            anchor = val_anchor[val_anchor.cutoff.ge(boundary) & val_anchor.cutoff.lt(end)].copy()
            joined = merge_predictions(anchor, read_xgb_archive(stage, boundary, end), 'xgb_all')
            val_parts.append(merge_predictions(joined, fit_five_starts(stage, boundary, end), 'xgb_five'))
        val = fixed_volume_mix(pd.concat(val_parts, ignore_index=True))
        val.to_parquet(RUN / 'val_predictions.parquet', index=False)
        selection = choose_on_val(val)
        with zipfile.ZipFile(SESSION_ARCHIVE) as archive:
            anchor = pd.read_parquet(BytesIO(archive.read('validation_predictions.parquet')))
            volume = pd.read_parquet(BytesIO(archive.read('evaluation_prediction_values.parquet')), columns=keys + ['calibrated_prediction'])
        check_panel(anchor, TEST_START, TEST_END)
        anchor = anchor.merge(volume.rename(columns={'calibrated_prediction': 'volume_model'}), on=keys, how='left', validate='one_to_one')
        joined = merge_predictions(anchor, read_xgb_archive('train_val', TEST_START, TEST_END), 'xgb_all')
        test = fixed_volume_mix(merge_predictions(joined, fit_five_starts('test', TEST_START, TEST_END), 'xgb_five'))
    test['selected_calibrated'] = test[selection['shape']] * selection['factor']
    columns = ('session_prediction', 'xgb_all', 'xgb_five', 'mix_all', 'mix_five', 'selected_calibrated')
    show_metrics(rows_for(test, 'test', columns), 'test_metrics.csv')
    rows = []
    for label, parts in [('horizon', test.groupby('horizon')), ('day', test.groupby('cutoff')), ('week', test.groupby(test.cutoff.dt.day.ge(25)))]:
        for value, part in parts:
            rows.extend(dict(slice=label, value=str(value), **row) for row in rows_for(part, 'test', columns))
    pd.DataFrame(rows).to_csv(RUN / 'test_slices.csv', index=False)
    test.to_parquet(RUN / 'test_predictions.parquet', index=False)
    (RUN / 'evaluation_note.txt').write_text('Локальный test оценён после выбора по val и до полного переобучения. Выборка ранее изучалась. На val используется восстановленный причинный Session6, на test архивный объём и прогноз Session6. Новый сабмит в этом опыте не собирается.')
    with zipfile.ZipFile(RUN.with_suffix('.zip'), 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=2) as archive:
        for path in sorted(RUN.rglob('*')):
            if path.is_file():
                archive.write(path, path.relative_to(RUN.parent))
    print((RUN / 'evaluation_note.txt').read_text())
    print(RUN.with_suffix('.zip'))


In [ ]:
from io import BytesIO
from tempfile import TemporaryDirectory
import lightgbm as lgb

STEP = pd.Timedelta(minutes=30)
ANCHOR_SEED = pd.Timestamp('2025-09-27')
VAL_FIRST = pd.Timestamp('2025-10-04')
VAL_SECOND = pd.Timestamp('2025-10-11')
MORNING_STARTS = (570, 600, 630, 660, 690)
keys = ['cutoff', 'horizon', 'route_id']
ACTIVE = EXPERIMENTS[0]
PIPELINE_VERSION = 'session6_xgb_5_vs_all_v1'
CODE_HASH = 'dc504bfb175cb9abcec80319462c3aadc090c040f47861656b4c490f47881163'
MODEL_CLUSTERS = (8,)
CAT_CONFIGS = [dict(experiment=ACTIVE, parameters=PARAMETERS, morning_starts=list(MORNING_STARTS))]
search = Path('/kaggle/input')


def locate(filename):
    matches = list(search.rglob(filename))
    if len(matches) != 1:
        raise FileNotFoundError('Подключите ровно один архив ' + filename)
    return matches[0]


SESSION_ARCHIVE = locate('cleaning_comparison_inputs.zip')
XGB_ARCHIVE = locate('cluster_experiment_4741653856669b1e.zip')
with TemporaryDirectory() as directory:
    with zipfile.ZipFile(SESSION_ARCHIVE) as archive:
        for filename in ('train_solo_track.parquet', 'test_solo_track.parquet'):
            (Path(directory) / filename).write_bytes(archive.read(filename))
    TRAIN_PATH = str(Path(directory) / 'train_solo_track.parquet')
    TEST_PATH = str(Path(directory) / 'test_solo_track.parquet')
    run_starts_comparison()
